# Unit 10. Autoregression and Forecasting

精算統計模型 (SOA Exam SRM/PA). The text of [unit10.pdf](https://github.com/chang-ye-tu/asm/blob/master/note/unit10.pdf) with every R chunk as a code cell.

<a href="https://colab.research.google.com/github/chang-ye-tu/asm/blob/master/colab/unit10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The notebook runs on the R runtime (on Colab: Runtime › Change runtime type › R). Run the setup cell first: it installs the packages and downloads the data this unit uses.

In [ ]:
# Setup: run this cell first. It installs the packages this unit uses and downloads
# its data files into data/, the paths the code below reads.
local({
  if (is.null(getOption("repos")) || identical(unname(getOption("repos")["CRAN"]), "@CRAN@"))
    options(repos = c(CRAN = "https://cloud.r-project.org"))
  os <- tryCatch(readLines("/etc/os-release"), error = function(e) character())
  code <- sub("^VERSION_CODENAME=", "", grep("^VERSION_CODENAME=", os, value = TRUE))
  if (length(code) == 1 && nzchar(code))  # Linux, e.g. Colab: prebuilt binary packages
    options(repos = c(CRAN = sprintf("https://packagemanager.posit.co/cran/__linux__/%s/latest", code)),
            HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(), paste(getRversion(),
              R.version$platform, R.version$arch, R.version$os)))
})
pkgs <- c("ISLR2", "tree")
new <- setdiff(pkgs, rownames(installed.packages()))
if (length(new)) install.packages(new)
data_files <- c("data/frees/csv/HKExchange.csv", "data/frees/csv/InflationBond.csv", "data/frees/csv/LaborForcePR.csv", "data/frees/csv/MedCPISmooth.csv", "data/frees/csv/PrescriptionDrug.csv", "data/frees/csv/SP500Daily.csv")
for (f in data_files) if (!file.exists(f)) {
  dir.create(dirname(f), recursive = TRUE, showWarnings = FALSE)
  url <- if (startsWith(f, "data/islr/"))
    paste0("https://www.statlearning.com/s/", basename(f)) else
    paste0("https://raw.githubusercontent.com/chang-ye-tu/asm/master/note/", f)
  download.file(url, f, quiet = TRUE, mode = "wb")
}
options(width = 62, digits = 4, scipen = 4, repr.plot.width = 6.4, repr.plot.height = 3.0)

In [ ]:
library(ISLR2)

## 1 Autoregressive Models of Order One

> **Reading.** FREES §8.2 (SRM reading list; not on the PA list). SRM learning outcomes 3(a), 3(b).

**Definition 10.1** (White noise and the innovations). Throughout (except in the section on ARCH/GARCH models), $\{\varepsilon_t\}_{t\in\mathbb Z}$ is independent and identically distributed with mean $0$ and variance $\sigma^2\in(0,\infty)$, and $\mathcal{L}_t$ is the prerequisites' **past space**: the closure in $L^2$ of the square-integrable functions of finitely many $\varepsilon_u$ with $u\leqslant t$. For $X\in L^2$, $\operatorname{\mathsf{E}}[X\mid\mathcal{L}_t]$ denotes the orthogonal projection of $X$ on the closed subspace $\mathcal{L}_t$: the nearest-point argument in the proof of the prerequisites' theorem **conditional expectation in $L^2$** applies to $\mathcal{L}_t$, and the error $X-\operatorname{\mathsf{E}}[X\mid\mathcal{L}_t]$ is orthogonal to $\mathcal{L}_t$.

**Definition 10.2** (Autoregressive model of order one; FREES (8.1)). The $\text{AR}(1)$ model is

$$
y_t=\beta_0+\beta_1y_{t-1}+\varepsilon_t , \tag{1}
$$

with $|\beta_1|<1$ and $\varepsilon_t$ independent of $\mathcal{L}_{t-1}$. FREES requires instead $\operatorname{cov}(\varepsilon_{t+k},y_t)=0$ for $k>0$, which follows because the solution lies in $\mathcal{L}_t$, as this section proves.

**Theorem 10.3** (Existence, uniqueness, moments; FREES §8.2, exercise 8.4). Let $|\beta_1|<1$ and $\mu=\beta_0/(1-\beta_1)$. Then

$$
y_t=\mu+\sum_{k=0}^{\infty}\beta_1^{k}\,\varepsilon_{t-k} \tag{2}
$$

converges in $L^2$, satisfies (1), and is the only weakly stationary solution of (1). It is weakly stationary with

$$
\operatorname{\mathsf{E}}[y_t]=\mu ,\qquad
\gamma(0)=\operatorname{var}(y_t)=\frac{\sigma^2}{1-\beta_1^2} ,\qquad
\gamma(k)=\beta_1^{k}\gamma(0) ,\qquad
\rho_k=\beta_1^{k} . \tag{3}
$$

**Proof.** The coefficients $\beta_1^k$ are square summable, so (2) converges in $L^2$ and lies in $\mathcal{L}_t$ by the prerequisites' construction of the past space. Splitting off the first term,

$$
y_t=\mu+\varepsilon_t+\beta_1\sum_{k=0}^{\infty}\beta_1^{k}\varepsilon_{t-1-k}
=\mu+\varepsilon_t+\beta_1(y_{t-1}-\mu) ,
$$

and $\mu-\beta_1\mu=\beta_0$, so (1) holds.

For the moments, $\operatorname{\mathsf{E}}[y_t]=\mu$ because each $\varepsilon$ has mean zero and $L^2$ convergence preserves the mean. The innovations are uncorrelated with common variance $\sigma^2$, so for $k\geqslant0$

$$
\gamma(k)=\operatorname{cov}\Bigl(\sum_{j\geqslant0}\beta_1^{j}\varepsilon_{t-j},\
                    \sum_{l\geqslant0}\beta_1^{l}\varepsilon_{t+k-l}\Bigr)
=\sigma^2\sum_{j\geqslant0}\beta_1^{j}\beta_1^{j+k}
=\sigma^2\beta_1^{k}\sum_{j\geqslant0}\beta_1^{2j}
=\frac{\sigma^2\beta_1^{k}}{1-\beta_1^{2}} ,
$$

the pairing $j=l-k$ being the only one that survives. Both moments are free of $t$.

For uniqueness, let $\{\tilde y_t\}$ be another weakly stationary solution and put $d_t=y_t-\tilde y_t$. Subtracting the two copies of (1) gives $d_t=\beta_1d_{t-1}=\beta_1^{m}d_{t-m}$ for every $m$. Weak stationarity bounds $\operatorname{\mathsf{E}}[d_{t-m}^2]$ by a constant $C$ free of $m$, so $\operatorname{\mathsf{E}}[d_t^2]\leqslant\beta_1^{2m}C\to0$, forcing $d_t=0$ almost surely. ∎

**Remark** (FREES §8.2). A correlogram decaying geometrically identifies an $\text{AR}(1)$ (FREES (8.2)); FREES calls a series with $\rho_1>0$ **meandering**. By (3), $\gamma(0)(1-\beta_1^2)=\sigma^2$: for $\beta_1\neq0$ the series is more variable than its innovations, increasingly so as $\beta_1\to1$.

**Proposition 10.4** (The two boundary cases; FREES §8.2). Setting $\beta_1=0$ in (1) gives white noise about $\beta_0$. Setting $\beta_1=1$ gives $\Delta y_t=\beta_0+\varepsilon_t$, a random walk with drift $\beta_0$, for which no weakly stationary solution exists.

**Proof.** The first is immediate. For the second, $\beta_1=1$ turns (1) into $y_t-y_{t-1}=\beta_0+\varepsilon_t$, which is Unit 9's random walk. If a weakly stationary solution existed, iterating would give $y_t-y_{t-m}=m\beta_0+\sum_{j<m}\varepsilon_{t-j}$ for every $m\geqslant1$, whose variance is $m\sigma^2$ because the $\varepsilon_t$ are iid; but for a weakly stationary series $\operatorname{var}(y_t-y_{t-m})=2\gamma(0)-2\gamma(m)\leqslant4\gamma(0)$ for every $m$, which is impossible since $\sigma^2>0$. ∎

**Proposition 10.5** (Fourth moment). If in addition $\mu_4=\operatorname{\mathsf{E}}[\varepsilon_t^4]<\infty$, then $x_t=y_t-\mu$ satisfies

$$
\operatorname{\mathsf{E}}[x_t^4]=\frac{\mu_4+6\beta_1^{2}\sigma^2\gamma(0)}{1-\beta_1^{4}}<\infty , \tag{4}
$$

and $\{x_t^2\}$ is weakly stationary with autocovariance $\operatorname{cov}(x_t^2,x_{t+k}^2)=\beta_1^{2k}\operatorname{var}(x_t^2)\to0$. Consequently $T^{-1}\sum_{t\leqslant T}x_t^2\xrightarrow{\,p\,}\gamma(0)$.

**Proof.** **Finiteness.** For the partial sum $S_m=\sum_{j<m}\beta_1^{j}\varepsilon_{t-j}$, expanding the fourth power and using independence with $\operatorname{\mathsf{E}}[\varepsilon]=0$ leaves only the terms in which every index occurs at least twice: the diagonal, and the three ways of splitting four factors into two distinct pairs. Hence

$$
\operatorname{\mathsf{E}}[S_m^4]=\mu_4\sum_{j<m}\beta_1^{4j}
 +3\sigma^4\Bigl\{\Bigl(\sum_{j<m}\beta_1^{2j}\Bigr)^{2}-\sum_{j<m}\beta_1^{4j}\Bigr\}
\ \leqslant\ \frac{\mu_4}{1-\beta_1^{4}}+\frac{3\sigma^4}{(1-\beta_1^{2})^{2}} ,
$$

a bound free of $m$. Now $S_m\to x_t$ in $L^2$, hence in probability and so almost surely along a subsequence, and Fatou's lemma gives $\operatorname{\mathsf{E}}[x_t^4]<\infty$. The law of $x_t$ is free of $t$, being the weak limit of the laws of the $S_m$, which are.

**Value.** Write $m_4=\operatorname{\mathsf{E}}[x_t^4]$ and use $x_t=\beta_1x_{t-1}+\varepsilon_t$ with $\varepsilon_t$ independent of $x_{t-1}$. The two cross terms with coefficient $4$ in the expansion carry a factor $\operatorname{\mathsf{E}}[\varepsilon_t]=0$ or $\operatorname{\mathsf{E}}[\varepsilon_t^3]\operatorname{\mathsf{E}}[x_{t-1}]=0$, and the term $6\beta_1^{2}x_{t-1}^2\varepsilon_t^2$ has expectation $6\beta_1^{2}\sigma^2\operatorname{\mathsf{E}}[x_{t-1}^2]$, so $m_4=\beta_1^{4}m_4+6\beta_1^{2}\sigma^2\operatorname{\mathsf{E}}[x_{t-1}^2]+\mu_4$, which is (4).

**Autocovariance.** Iterate (1) $k$ times to get $x_{t+k}=\beta_1^{k}x_t+v$ with $v=\sum_{j<k}\beta_1^{j}\varepsilon_{t+k-j}$ independent of $x_t$ and of mean zero. Then $x_{t+k}^2=\beta_1^{2k}x_t^2+2\beta_1^{k}x_tv+v^2$, and $\operatorname{cov}(x_t^2,x_tv)=\operatorname{\mathsf{E}}[x_t^3]\operatorname{\mathsf{E}}[v]-\operatorname{\mathsf{E}}[x_t^2]\operatorname{\mathsf{E}}[x_t]\operatorname{\mathsf{E}}[v]=0$ while $\operatorname{cov}(x_t^2,v^2)=0$, both by independence. So $\operatorname{cov}(x_t^2,x_{t+k}^2)=\beta_1^{2k}\operatorname{var}(x_t^2)$, which tends to $0$, and the prerequisites' lemma **variance of a stationary average** applies. ∎

**Remark** (FREES §8.2; SOA sample questions 22 and 64). Stationarity restricts $\beta_1$ to $(-1,1)$ and leaves $\beta_0$ free (Theorem 10.3); $\beta_1=0$ is white noise and $\beta_1=1$ a random walk (Proposition 10.4).

**Theorem 10.6** (Joint limit of the sample autocorrelations; FREES §8.2). Let $y_1,\ldots,y_T$ be white noise with mean $\mu$, variance $\sigma^2$ and $\operatorname{\mathsf{E}} y_t^4<\infty$, and fix $K\geqslant1$. Then

$$
\sqrt{T}\,(r_1,\ldots,r_K)'\ \xrightarrow{\,d\,}\ N(\mathbf{0},\ \mathbf{I}_K) . \tag{5}
$$

**Proof.** Put $\xi_t=y_t-\mu$ and $\mathbf{a}_j=\mathbf{e}_{j+1}\in\mathbb{R}^K$ for $0\leqslant j\leqslant K-1$, $\mathbf{a}_j=\mathbf{0}$ beyond, so that $\mathbf{u}_{t-1}=\sum_j\mathbf{a}_j\xi_{t-1-j} =(\xi_{t-1},\ldots,\xi_{t-K})'$ and $\boldsymbol{\Gamma}=\operatorname{\mathsf{E}}[\mathbf{u}\mathbf{u}']=\sigma^2\mathbf{I}_K$. The prerequisites' central limit theorem for autoregressive scores gives

$$
\frac1{\sqrt T}\sum_{t=K+1}^{T}\mathbf{u}_{t-1}\xi_t\ \xrightarrow{\,d\,}\ N(\mathbf{0},\ \sigma^4\mathbf{I}_K) ,
$$

the theorem's sum running from $t=1$ and this one from $t=K+1$, a difference of $K$ terms $T^{-1/2}\mathbf{u}_{t-1}\xi_t$, each of which tends to $0$ in probability by Chebyshev's inequality, its expected squared norm being $K\sigma^4/T$.

The $k$th coordinate of the sum is $\sum_t\xi_t\xi_{t-k}$, the numerator of $r_k$ with the mean known, up to the $K-k$ terms with $k<t\leqslant K$, which after division by $\sqrt T$ tend to $0$ in probability by the same Chebyshev argument. Writing $\bar\xi=\bar y-\mu$, the numerator actually used differs from it by

$$
-\bar\xi\sum_{t}\xi_{t-k}-\bar\xi\sum_{t}\xi_{t}+(T-k)\bar\xi^{2} ,
$$

and after division by $\sqrt T$ each of the three terms is the product of $\sqrt T\,\bar\xi$, which converges in distribution to $N(0,\sigma^2)$ by the prerequisites' central limit theorem for independent summands, with a factor that tends to $0$ in probability ($T^{-1}\sum_t\xi_{t-k}$, $T^{-1}\sum_t\xi_t$ or $(T-k)T^{-1}\bar\xi$, by the lemma cited next), so each tends to $0$ in probability by Slutsky's theorem. The denominator satisfies $T^{-1}\sum_t(y_t-\bar y)^2=T^{-1}\sum_t\xi_t^2-\bar\xi^2\xrightarrow{\,p\,}\sigma^2$, by the lemma **variance of a stationary average** of the prerequisites applied to the iid sequence $\{\xi_t^2\}$, which is square integrable because $\operatorname{\mathsf{E}} y_t^4<\infty$, and to $\{\xi_t\}$, which gives $\bar\xi\xrightarrow{\,p\,}0$. Dividing and applying Slutsky's theorem gives (5). ∎

**Corollary 10.7** (The correlogram band). Under white noise with $\operatorname{\mathsf{E}} y_t^4<\infty$, $\operatorname{\mathsf{P}}\{|r_k|>2/\sqrt T\}\to\operatorname{\mathsf{P}}\{|Z|>2\}=0.0455$ for each fixed $k$, $Z$ being standard normal.

**Proof.** $\sqrt T\,r_k\xrightarrow{\,d\,} N(0,1)$ by Theorem 10.6, and $\pm2$ are continuity points of the normal distribution function. ∎

**Remark** (FREES §8.2). FREES's standard error $\operatorname{se}(r_k)=1/\sqrt T$ under no autocorrelation and its rule “$|r_k|>2/\sqrt T$” are Corollary 10.7: a test of level $4.55\%$ in the limit.

**Example 10.1** (Inflation bond index; FREES §8.2, Table 8.4). FREES's $T=51$ monthly values (Unit 9):

In [ ]:
yb <- read.csv("data/frees/csv/InflationBond.csv")$INFBOND; Tb <- length(yb)
rb <- acf(yb, lag.max = 5, plot = FALSE)$acf[2:6]
round(rbind(r_k = rb, ar1_r1_power_k = rb[1]^(1:5)), 3)
round(c(se = 1 / sqrt(Tb), r1_over_se = rb[1] * sqrt(Tb)), 3)

$r_1=0.814$ is $5.81$ standard errors from $0$, so white noise is rejected. Under an $\text{AR}(1)$ the estimates $r_1^k$ of $\rho_k$ track the $r_k$ within $0.03$ at every lag; FREES's Table 8.4 lists $r_5=0.267$, which is $0.367$ (Unit 9).

## 2 Estimation and Diagnostic Checking

> **Reading.** FREES §8.3. SRM learning outcomes 3(b), 3(c).

**Definition 10.8** (Conditional least squares; FREES §8.3). Given $y_1,\ldots,y_T$, the **conditional least squares** estimates $b_0,b_1$ minimise $\sum_{t=2}^{T}(y_t-b_0-b_1y_{t-1})^2$: least squares of $y_t$ on $y_{t-1}$ over the $T-1$ usable dates. The residuals are $e_t=y_t-(b_0+b_1y_{t-1})$ and

$$
s^2=\frac{1}{T-3}\sum_{t=2}^{T}(e_t-\bar e)^2 . \tag{6}
$$

**Proposition 10.9** (Exact and approximate estimates; FREES §8.3, exercise 8.3). With $\bar y_-=(T-1)^{-1}\sum_{t=1}^{T-1}y_t$ and $\bar y_+=(T-1)^{-1}\sum_{t=2}^{T}y_t$,

$$
b_1=\frac{\sum_{t=2}^{T}(y_{t-1}-\bar y_-)(y_t-\bar y_+)}{\sum_{t=2}^{T}(y_{t-1}-\bar y_-)^2} ,
\qquad b_0=\bar y_+-b_1\bar y_- , \tag{7}
$$

and $\sum_te_t=0$, so $\bar e=0$ and (6) is the residual sum of squares over $T-3$. If $y_1=y_T=\bar y$, then $b_1=r_1$ and $b_0=\bar y(1-r_1)$; in general both differences are end effects (not quantified here).

**Proof.** The formulas are Unit 2's simple regression applied to the $T-1$ pairs $(y_{t-1},y_t)$, whose residuals sum to zero because of the intercept. If $y_1=y_T=\bar y$, then $\bar y_-=(T\bar y-y_T)/(T-1)=\bar y$ and likewise $\bar y_+=(T\bar y-y_1)/(T-1)=\bar y$, so the numerator of $b_1$ is $\sum_{t\geqslant2}(y_{t-1}-\bar y)(y_t-\bar y)$, the numerator of $r_1$, and its denominator $\sum_{t\leqslant T-1}(y_t-\bar y)^2=\sum_{t\leqslant T}(y_t-\bar y)^2$, that of $r_1$; then $b_0=\bar y-r_1\bar y$. ∎

**Remark** (FREES (8.3)). FREES's $\bar e$ in (6), “because without the first residual the average of the residuals is no longer automatically zero”, is $0$ for the least squares fit by Proposition 10.9; it is not zero for the approximation $b_1=r_1$, $b_0=\bar y(1-r_1)$, which is what SOA sample question 58 uses.

**Theorem 10.10** (Asymptotics of conditional least squares; FREES §8.3). Let $\{y_t\}$ be the stationary $\text{AR}(1)$ of Theorem 10.3 with $\operatorname{\mathsf{E}}[\varepsilon_t^4]<\infty$. Then, as $T\to\infty$,

$$
b_1\xrightarrow{\,p\,}\beta_1 ,
\qquad
\sqrt{T}\,(b_1-\beta_1)\ \xrightarrow{\,d\,}\ N\bigl(0,\ 1-\beta_1^2\bigr) . \tag{8}
$$

**Proof.** Write $x_t=y_t-\mu$, so that (7) becomes, after subtracting $\mu$ from every observation,

$$
b_1-\beta_1
=\frac{\sum_{t=2}^{T}(x_{t-1}-\bar x_-)\varepsilon_t}{\sum_{t=2}^{T}(x_{t-1}-\bar x_-)^2} ,
$$

because $x_t=\beta_1x_{t-1}+\varepsilon_t$ makes the numerator of (7) equal to $\beta_1\sum(x_{t-1}-\bar x_-)^2+\sum(x_{t-1}-\bar x_-)\varepsilon_t$, the centring of the second factor contributing nothing since $\sum_t(x_{t-1}-\bar x_-)=0$.

**Denominator.** $\{x_t\}$ is weakly stationary with $\gamma(k)=\beta_1^k\gamma(0)\to0$, so $\bar x_-\xrightarrow{\,p\,}0$ by the prerequisites' lemma **variance of a stationary average**, and $\{x_t^2\}$ is weakly stationary with vanishing autocovariance by Proposition 10.5, so $T^{-1}\sum_{t}x_{t-1}^2\xrightarrow{\,p\,}\gamma(0)$. Hence

$$
\frac1T\sum_{t=2}^{T}(x_{t-1}-\bar x_-)^2
=\frac1T\sum_{t=2}^{T}x_{t-1}^2-\frac{T-1}{T}\bar x_-^{2}\ \xrightarrow{\,p\,}\ \gamma(0)>0 .
$$

**Numerator.** With $a_j=\beta_1^{j}$ the sequence $x_{t-1}=\sum_{j\geqslant0} a_j\varepsilon_{t-1-j}$ is exactly the linear combination of the prerequisites' central limit theorem for autoregressive scores, whose limit variance is $\sigma^2\cdot\sigma^2\sum_j\beta_1^{2j}=\sigma^2\gamma(0)$. So $T^{-1/2}\sum_{t}x_{t-1}\varepsilon_t\xrightarrow{\,d\,} N(0,\sigma^2\gamma(0))$, the theorem's sum running from $t=1$ and this one from $t=2$, a difference $x_0\varepsilon_1/\sqrt T$ that tends to $0$ in probability by Chebyshev's inequality. The centring costs nothing:

$$
\frac1{\sqrt T}\sum_{t=2}^{T}(x_{t-1}-\bar x_-)\varepsilon_t
=\frac1{\sqrt T}\sum_{t=2}^{T}x_{t-1}\varepsilon_t
 -\bar x_-\cdot\frac1{\sqrt T}\sum_{t=2}^{T}\varepsilon_t ,
$$

and the second term is a product of $\bar x_-\xrightarrow{\,p\,}0$ with a quantity converging in distribution to $N(0,\sigma^2)$ by the prerequisites' central limit theorem for independent summands, hence tends to $0$ in probability by Slutsky's theorem.

**Ratio.** Slutsky's theorem again gives $\sqrt T(b_1-\beta_1)\xrightarrow{\,d\,} N(0,\sigma^2\gamma(0)/\gamma(0)^2)$, and $\sigma^2/\gamma(0)=1-\beta_1^2$ by (3). Consistency follows from the same two limits with the $\sqrt T$ removed.

**Intercept and residual variance.** Since $\beta_0=\mu(1-\beta_1)$, (7) gives $b_0-\beta_0=\bar x_+-b_1\bar x_--\mu(b_1-\beta_1)$, with $\bar x_\pm=\bar y_\pm-\mu$. By the lemma **variance of a stationary average**, $T\operatorname{var}(\bar x_\pm)\leqslant c\sum_k|\gamma(k)|<\infty$ for a constant $c$ not depending on $T$, so by Chebyshev's inequality $\operatorname{\mathsf{P}}\{\sqrt T|\bar x_\pm|>M\}\leqslant c\sum_k|\gamma(k)|/M^2$ for every $T$; together with $b_1\xrightarrow{\,p\,}\beta_1$ and (8), this gives $b_0\xrightarrow{\,p\,}\beta_0$ and makes $\operatorname{\mathsf{P}}\{\sqrt T|b_0-\beta_0|>M\}$ small, uniformly in large $T$, for $M$ large: the error in $b_0$ is of order $T^{-1/2}$, as is that in $b_1$. Next, $e_t=\varepsilon_t-(b_0-\beta_0)-(b_1-\beta_1)y_{t-1}$, so $T^{-1}\sum_te_t^2-T^{-1}\sum_t\varepsilon_t^2$ is a sum of products, each of a factor $b_0-\beta_0$ or $b_1-\beta_1$ (which tend to $0$ in probability) with one of, or a product of two of, $b_0-\beta_0$, $b_1-\beta_1$, $T^{-1}\sum_t\varepsilon_t$, $T^{-1}\sum_ty_{t-1}$, $T^{-1}\sum_ty_{t-1}\varepsilon_t$ and $T^{-1}\sum_ty_{t-1}^2$ (each of which converges in probability to a constant, by the limits above); hence it tends to $0$ in probability. Finally $T^{-1}\sum_t\varepsilon_t^2\xrightarrow{\,p\,}\sigma^2$ by the same lemma applied to the iid, square-integrable $\{\varepsilon_t^2\}$, and $T/(T-3)\to1$, so $s^2\xrightarrow{\,p\,}\sigma^2$. ∎

**Corollary 10.11** (Standard error). An approximate standard error for $b_1$ is $\sqrt{(1-b_1^2)/T}$, and its ratio to the ordinary least squares standard error $s/\{\sum_t(y_{t-1}-\bar y_-)^2\}^{1/2}$ reported by a regression routine tends to $1$ in probability.

**Proof.** The first expression is (8) with $\beta_1$ estimated. For the second, the regression standard error squared is $s^2/\sum_t(y_{t-1}-\bar y_-)^2$, whose numerator converges in probability to $\sigma^2$ (proof of Theorem 10.10) and whose denominator is $T$ times a quantity converging to $\gamma(0)$; the ratio times $T$ therefore tends to $\sigma^2/\gamma(0)=1-\beta_1^2$, which is also the limit of $T$ times the square of the first expression. The ratio of the two standard errors therefore tends to $1$. ∎

**Example 10.2** (Inflation bond index, fitted; FREES §8.3, Table 8.5).

In [ ]:
mb <- lm(yb[-1] ~ yb[-Tb]); eb <- residuals(mb)
round(coef(summary(mb))[, 1:3], 4)
round(c(s = sqrt(sum((eb - mean(eb))^2) / (Tb - 3)), se_asymptotic = sqrt((1 - coef(mb)[[2]]^2) / Tb),
        b0_approx = mean(yb) * (1 - rb[1])), 4)
am <- arima(yb, order = c(1, 0, 0), method = "ML")
round(c(ml_ar1 = coef(am)[["ar1"]], ml_b0 = coef(am)[["intercept"]] * (1 - coef(am)[["ar1"]])), 4)
round(rbind(cls_residuals = acf(eb, lag.max = 5, plot = FALSE)$acf[2:6],
            ml_residuals = acf(residuals(am), lag.max = 5, plot = FALSE)$acf[2:6]), 3)

Conditional least squares gives $0.4085+0.8138\,y_{t-1}$, standard errors $0.1692$ and $0.0749$, $s=0.137$. FREES reports $0.2923+0.8727\,y_{t-1}$ with standard errors $0.0196$ and $0.0736$ and $s=0.14$; neither conditional least squares nor exact maximum likelihood ($0.3347+0.8538\,y_{t-1}$) reproduces it, and an intercept standard error of $0.0196$ is not possible here, the series having mean $2.245$ and the slope standard error $0.0736$. FREES's residual autocorrelations ($0.09$, $-0.33$, $0.07$, $0.02$, $-0.17$) are close to those of the maximum likelihood fit. In both, $r_2(e)$ is about $-2$ standard errors $1/\sqrt{51}=0.14$. Figure 10.1 shows the fitted line and the control chart of the residuals.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
op <- par(mfrow = c(1, 2), mar = c(3.4, 3.8, 1.4, 0.6), mgp = c(2.1, 0.6, 0), cex = 0.8)
plot(yb[-Tb], yb[-1], pch = 16, cex = 0.6, xlab = expression(y[t - 1]), ylab = expression(y[t]),
     main = "index against lagged index"); abline(mb, col = "#B45309", lwd = 2)
plot(2:Tb, eb, type = "o", pch = 16, cex = 0.5, xlab = "t", ylab = "residual",
     main = "control chart of residuals", ylim = range(eb, mean(eb) + c(-3, 3) * sd(eb)))
abline(h = mean(eb) + c(-3, 0, 3) * sd(eb), lty = c(2, 1, 2), col = "#B45309")
par(op)

*Figure 10.1.* Inflation bond index (FREES Figures 8.2 and 8.4): the index against its lagged value with the conditional least squares line, and the control chart of the residuals.

**Remark** (Residual autocorrelations; FREES §8.3, §8.6). Under a fitted $\text{AR}(1)$ the standard error $1/\sqrt T$ of $r_k(e)$ is only approximate; for $k=1$ it is nearer $|\beta_1|/\sqrt T$ (FREES §8.6, citing MacLeod), and that refinement is not proved here.

**Definition 10.12** (Portmanteau statistics (beyond the syllabus); FREES §8.6). For residual autocorrelations $r_k(e)$ and a user-chosen $K$,

$$
Q_{\text{BP}}=T\sum_{k=1}^{K}r_k(e)^2 ,
\qquad
Q_{\text{LB}}=T(T+2)\sum_{k=1}^{K}\frac{r_k(e)^2}{T-k} . \tag{9}
$$

**Theorem 10.13** (Limit for an unfitted series (beyond the syllabus)). If $y_1,\ldots,y_T$ is white noise with $\operatorname{\mathsf{E}} y_t^4<\infty$ and the $r_k$ are computed from the series itself, then $Q_{\text{BP}}\xrightarrow{\,d\,}\chi^2_K$ and $Q_{\text{LB}}\xrightarrow{\,d\,}\chi^2_K$.

**Proof.** $Q_{\text{BP}}=|\sqrt T\,\mathbf{r}|^2$ with $\mathbf{r}=(r_1,\ldots,r_K)'$, and Theorem 10.6 with the continuous mapping theorem gives $|\sqrt T\mathbf{r}|^2\xrightarrow{\,d\,}|Z|^2$ for $Z\sim N(\mathbf{0},\mathbf{I}_K)$, which is $\chi^2_K$ by the prerequisites. For $Q_{\text{LB}}$, write

$$
Q_{\text{LB}}=\sum_{k=1}^{K}\frac{T(T+2)}{T-k}\,r_k^2
      =\sum_{k=1}^{K}\Bigl(1+\frac{k+2}{T-k}\Bigr)T\,r_k^2 ,
$$

so $Q_{\text{LB}}-Q_{\text{BP}}=\sum_k\{(k+2)/(T-k)\}Tr_k^2\xrightarrow{\,p\,}0$, because $Tr_k^2\xrightarrow{\,d\,}\chi^2_1$ by Theorem 10.6 and the continuous mapping theorem while $(k+2)/(T-k)\to0$, so each product tends to $0$ in probability by Slutsky's theorem; Slutsky's theorem finishes. ∎

**Remark**. When the $r_k$ are computed from the residuals of a fitted $\text{ARMA}(p,q)$, the limit is $\chi^2_{K-p-q}$ rather than $\chi^2_K$: fitting removes exactly $p+q$ directions from the residual correlogram. That statement is **not** proved here. It needs the joint limit of the residual autocorrelations, which involves the estimation error and is beyond the prerequisites as they stand. For the $\text{AR}(1)$ it is measured by simulation in this unit instead: the mean of $Q_{\text{LB}}$ sits near $K-1=K-p-q$, and the size is the nominal $5\%$ only when the degrees of freedom are reduced.

**Remark** (FREES §8.6). FREES refers $Q_{\text{BP}}$ and $Q_{\text{LB}}$ computed from the residuals of a fitted $\text{ARMA}(p,q)$ to $\chi^2$ with $K$ minus the number of linear parameters, $1+p+q$; the usual reference subtracts $p+q$. Neither statement for fitted residuals is proved here: it needs the joint law of the residual autocorrelations, which carries the estimation error.

## 3 Smoothing and Prediction

> **Reading.** FREES §8.4. SRM learning outcomes 3(b), 3(c).

**Definition 10.14** (Smoothed series and chain rule; FREES §8.4, (8.4)). The **smoothed series** is $\hat y_t=b_0+b_1y_{t-1}$. The $k$-step forecast is $\hat y_{T+k}=b_0+b_1\hat y_{T+k-1}$, starting from $\hat y_T=y_T$: the **chain rule of forecasting**.

**Theorem 10.15** (Optimal forecast and its error; FREES §8.4, exercise 8.5). For the stationary $\text{AR}(1)$ with known parameters, the minimum mean squared error predictor of $y_{T+l}$ among all square-integrable functions of the past is

$$
\operatorname{\mathsf{E}}[y_{T+l}\mid \mathcal{L}_T]=\mu+\beta_1^{l}(y_T-\mu) , \tag{10}
$$

and its error variance is

$$
\operatorname{var}(y_{T+l}-\operatorname{\mathsf{E}}[y_{T+l}\mid \mathcal{L}_T])=\sigma^2\sum_{k=0}^{l-1}\beta_1^{2k}
=\sigma^2\,\frac{1-\beta_1^{2l}}{1-\beta_1^{2}} . \tag{11}
$$

The point forecast tends to $\mu$ and the error variance to $\gamma(0)$ as $l\to\infty$.

**Proof.** Iterating (1) forward $l$ steps,

$$
y_{T+l}-\mu=\beta_1^{l}(y_T-\mu)+\sum_{k=0}^{l-1}\beta_1^{k}\varepsilon_{T+l-k} . \tag{12}
$$

Each innovation in the sum has index exceeding $T$, hence is independent of $\mathcal{L}_T$ and has mean zero, while $y_T-\mu\in\mathcal{L}_T$; taking conditional expectations gives (10), which minimises mean squared error by the projection property of conditional expectation in $L^2$ (prerequisites). The forecast error is the sum in (12), whose terms are uncorrelated with variances $\sigma^2\beta_1^{2k}$, giving (11). The limits follow from $|\beta_1|<1$ and (3). ∎

**Corollary 10.16** (Forecast interval). If moreover the $\varepsilon_t$ are normal, an approximate $100(1-\alpha)\%$ interval for $y_{T+l}$ is

$$
\hat y_{T+l}\ \pm\ t_{T-3,\,1-\alpha/2}\;s\;
\Bigl(\sum_{k=0}^{l-1}b_1^{2k}\Bigr)^{1/2} , \tag{13}
$$

which for $l=1$ is $\hat y_{T+1}\pm t\,s$ and for $l=2$ is $\hat y_{T+2}\pm t\,s(1+b_1^2)^{1/2}$.

**Proof.** With known parameters the forecast error is the sum in (12), a fixed linear combination of $l$ innovations; for normal $\varepsilon_t$ it is $N\bigl(0,\sigma^2\sum_{k<l}\beta_1^{2k}\bigr)$, so the interval with $\beta_0,\beta_1,\sigma$ in place of $b_0,b_1,s$ and the normal quantile $z_{1-\alpha/2}$ in place of $t$ covers $y_{T+l}$ with probability exactly $1-\alpha$. Substitute the estimates into (11) and treat the estimation error in $b_0,b_1$ as negligible: by Theorem 10.10 and its proof, $b_0,b_1$ err by order $T^{-1/2}$ and $s\xrightarrow{\,p\,}\sigma$, so the estimated centre differs from the known-parameter one by a term tending to $0$ in probability, the half-width converges in probability to the known-parameter one, and $t_{T-3,1-\alpha/2}\to z_{1-\alpha/2}$; by Slutsky's theorem the coverage tends to $1-\alpha$. Without normality the law of the error does not depend on $T$ and is not normal in general, so the coverage need not approach $1-\alpha$. ∎

**Remark** (FREES §8.4). FREES's interval is $\hat y_{T+k}\pm t\,s\{1+b_1^2+\cdots+b_1^{2(k-1)}\}^{1/2}$ with $t$ from the $t_{T-3}$ law, “the percentile $1-(\text{prediction level})/2$”; the percentile is $1-(1-\text{level})/2$, that is $0.975$ for a $95\%$ interval.

**Example 10.3** (Forecasting the inflation bond index; FREES §8.4, Figure 8.6).

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
b0b <- coef(mb)[[1]]; b1b <- coef(mb)[[2]]; sb <- summary(mb)$sigma
fcb <- Reduce(function(prev, k) b0b + b1b * prev, 1:6, accumulate = TRUE, init = yb[Tb])[-1]
round(cbind(k = 1:6, forecast = fcb,
            half_width = qt(0.975, Tb - 3) * sb * sqrt(cumsum(b1b^(2 * (0:5)))),
            limit_mean = b0b / (1 - b1b), limit_half = qt(0.975, Tb - 3) * sb / sqrt(1 - b1b^2)), 4)

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
hw <- qt(0.975, Tb - 3) * sb * sqrt(cumsum(b1b^(2 * (0:5))))
par(mar = c(4.2, 4, 0.5, 0.5), cex = 0.8)
plot(1:Tb, yb, type = "l", xlim = c(1, Tb + 6), ylim = range(yb, fcb - hw, fcb + hw),
     xlab = "month", ylab = "inflation bond index")
lines(Tb + 1:6, fcb, lwd = 2, col = "#B45309")
lines(Tb + 1:6, fcb - hw, lty = 2, col = "#B45309"); lines(Tb + 1:6, fcb + hw, lty = 2, col = "#B45309")

From $y_{51}=2.25$ the forecasts move towards $b_0/(1-b_1)=2.194$, and the half-widths grow towards $t\,s/\sqrt{1-b_1^2}$, the limits of Theorem 10.15. The plot, FREES's Figure 8.6, shows the series with the six forecasts and their $95\%$ intervals.

## 4 Box–Jenkins Models (beyond the syllabus)

> **Reading.** Beyond the syllabus: FREES §8.5–8.6, outside the SRM list (chapter 8, sections 1–4), and ISLR §10.5.2 with the lab §10.9.6, on neither list.

**Definition 10.17** (Backshift; FREES §8.5.1). $\mathrm{B}y_t=y_{t-1}$ and $\mathrm{B}^ky_t=y_{t-k}$. For a polynomial $\Phi(x)=1-\beta_1x-\cdots-\beta_px^p$, $\Phi(\mathrm{B})y_t =y_t-\beta_1y_{t-1}-\cdots-\beta_py_{t-p}$.

**Definition 10.18** ($\text{AR}(p)$, $\text{MA}(q)$, $\text{ARMA}$, $\text{ARIMA}$; FREES (8.5)–(8.10)). With $\Phi(x)=1-\beta_1x-\cdots-\beta_px^p$ and $\Theta(x)=1-\theta_1x-\cdots-\theta_qx^q$,

$$
\begin{align}
\text{AR}(p):&\quad \Phi(\mathrm{B})y_t=\beta_0+\varepsilon_t , \tag{14} \\
\text{MA}(q):&\quad y_t=\beta_0+\Theta(\mathrm{B})\varepsilon_t , \tag{15} \\
\text{ARMA}(p,q):&\quad \Phi(\mathrm{B})y_t=\beta_0+\Theta(\mathrm{B})\varepsilon_t , \tag{16}
\end{align}
$$

and $\text{ARIMA}(p,d,q)$ is $\text{ARMA}(p,q)$ applied to $w_t=(1-\mathrm{B})^dy_t$.

**Proposition 10.19** ($\text{MA}(q)$ is always stationary and its correlogram cuts off). The $\text{MA}(q)$ process (15) is weakly stationary for every $\theta_1,\ldots,\theta_q$, with $\theta_0=-1$ and

$$
\gamma(k)=\sigma^2\sum_{j=0}^{q-k}\theta_j\theta_{j+k}\ \ (0\leqslant k\leqslant q) ,
\qquad \gamma(k)=0 \ \ (k>q) . \tag{17}
$$

**Proof.** $y_t-\beta_0$ is a finite linear combination of $\varepsilon_t,\ldots,\varepsilon_{t-q}$ with coefficients $-\theta_j$ and $\theta_0=-1$, so the mean is $\beta_0$ and the covariance is $\sigma^2\sum_j\theta_j\theta_{j+k}$ over those $j$ with both indices in $\{0,\ldots,q\}$, all free of $t$. For $k>q$ the two windows $\{t-q,\ldots,t\}$ and $\{t+k-q,\ldots,t+k\}$ are disjoint and the innovations independent. ∎

**Theorem 10.20** (Stationarity of $\text{AR}(p)$). Suppose $\Phi$ has $p$ distinct roots $g_1,\ldots,g_p$, all with $|g_i|>1$. Then $1/\Phi(x)=\sum_{k\geqslant0}\psi_kx^k$ for $|x|<\min_i|g_i|$, with

$$
\psi_k=\sum_{i=1}^{p}c_ig_i^{-k} ,
\qquad c_i=\prod_{l\neq i}\bigl(1-g_i/g_l\bigr)^{-1} , \tag{18}
$$

and $|\psi_k|\leqslant\bigl(\sum_i|c_i|\bigr)\varrho^{k}$ with $\varrho=\max_i|g_i|^{-1}<1$. Consequently $y_t=\mu+\sum_{k\geqslant0}\psi_k\varepsilon_{t-k}$, with $\mu=\beta_0/\Phi(1)$, converges in $L^2$ and is the unique weakly stationary solution of (14).

**Proof.** Since the roots are distinct, $\Phi(x)=\prod_i(1-x/g_i)$ and the partial fraction expansion $1/\Phi(x)=\sum_ic_i/(1-x/g_i)$ holds with $c_i$ obtained by multiplying through by $(1-x/g_i)$ and setting $x=g_i$, which gives $c_i=\prod_{l\neq i}(1-g_i/g_l)^{-1}$. Expanding each factor as a geometric series, valid for $|x|<|g_i|$, and collecting powers gives (18) and the bound.

Absolute summability of $\{\psi_k\}$ makes the series square summable, so it converges in $L^2$. Applying $\Phi(\mathrm{B})$ term by term, the coefficient of $\varepsilon_{t-k}$ in $\Phi(\mathrm{B})\sum_j\psi_j\varepsilon_{t-j}$ is the $k$th coefficient of $\Phi(x)\sum_j\psi_jx^j=1$, hence $1$ for $k=0$ and $0$ otherwise; the interchange is legitimate because $\Phi(\mathrm{B})$ is a finite linear combination and $L^2$ limits are preserved by such. So $\Phi(\mathrm{B})(y_t-\mu)=\varepsilon_t$, and $\Phi(\mathrm{B})\mu=\mu\Phi(1)=\beta_0$.

For uniqueness, let $d_t$ be the difference of two weakly stationary solutions, so $\Phi(\mathrm{B})d_t=0$ and $\operatorname{\mathsf{E}}[d_t^2]\leqslant C$ free of $t$. Multiplying the truncated series $\sum_{k\leqslant K}\psi_kx^k$ by $\Phi(x)$ leaves $1$ plus terms in $x^{K+1},\ldots,x^{K+p}$ whose coefficients are sums of at most $p$ products $\psi_k\beta_j$ with $K-p<k\leqslant K$ (the coefficient of $x^m$ is $-\sum_{m-p\leqslant k\leqslant K}\psi_k\beta_{m-k}$), hence bounded by $p\max_j|\beta_j|\sum_i|c_i|\varrho^{K-p+1}=C'\varrho^{K}$ with $C'=p\max_j|\beta_j|\sum_i|c_i|\varrho^{1-p}$. Applying the same identity with $\mathrm{B}$ in place of $x$ to $d_t$,

$$
0=\sum_{k\leqslant K}\psi_k\,\Phi(\mathrm{B})d_{t-k}
 =d_t+\sum_{j=K+1}^{K+p}e_{K,j}\,d_{t-j} ,
\qquad |e_{K,j}|\leqslant C'\varrho^{K} ,
$$

so $\operatorname{\mathsf{E}}[d_t^2]^{1/2}\leqslant pC'\varrho^{K}C^{1/2}\to0$ as $K\to\infty$ by the triangle inequality in $L^2$, forcing $d_t=0$ almost surely. ∎

**Remark**. For $p=1$ the condition is $|g_1|=|\beta_1|^{-1}>1$, that is $|\beta_1|<1$. The requirement is on the roots, not on the coefficients: an $\text{AR}(2)$ with $\beta_1=1.02$ and $\beta_2=-0.24$ is stationary, because $\Phi(x)=1-1.02x+0.24x^2$ has both roots outside the unit circle, while its first coefficient exceeds one.

**Definition 10.21** ($\psi$-coefficients). The representation $y_t=\mu+\sum_{k\geqslant0}\psi_k\varepsilon_{t-k}$ with $\psi_0=1$ is the **$\psi$-coefficient representation**. For $\text{AR}(1)$, $\psi_k=\beta_1^{k}$; for $\text{MA}(q)$, $\psi_k=-\theta_k$ for $1\leqslant k\leqslant q$ and $0$ beyond.

### 4.1 Partial autocorrelation

**Definition 10.22** (Partial autocorrelation). For a weakly stationary series for which the covariance matrix of $(y_{t-1},\ldots,y_{t-k})$ is nonsingular, let $(\alpha_{1,k},\ldots,\alpha_{k,k})$ be the (unique) coefficients of the projection of $y_t-\mu$ on $\{y_{t-1}-\mu,\ldots,y_{t-k}-\mu\}$, that is the minimiser of $\operatorname{\mathsf{E}}[\{y_t-\mu-\sum_{j\leqslant k}a_j(y_{t-j}-\mu)\}^2]$. The **partial autocorrelation** at lag $k$ is $\alpha_{k,k}$.

**Theorem 10.23** (The partial correlogram of an $\text{AR}(p)$ cuts off). Let $\{y_t\}$ be the stationary $\text{AR}(p)$ of Theorem 10.20. Then $(\alpha_{1,k},\ldots,\alpha_{k,k})=(\beta_1,\ldots,\beta_p,0,\ldots,0)$ for every $k\geqslant p$. In particular $\alpha_{p,p}=\beta_p$ and $\alpha_{k,k}=0$ for $k>p$.

**Proof.** Write $x_t=y_t-\mu$, so $x_t=\sum_{j\leqslant p}\beta_jx_{t-j}+\varepsilon_t$. Fix $k\geqslant p$ and let $a=(\beta_1,\ldots,\beta_p,0,\ldots,0)$. The residual of that candidate is exactly $\varepsilon_t$, and $x_{t-j}\in\mathcal{L}_{t-1}$ for $j\geqslant1$ while $\varepsilon_t$ is independent of $\mathcal{L}_{t-1}$ with mean zero, so $\operatorname{\mathsf{E}}[\varepsilon_tx_{t-j}]=0$ for $j=1,\ldots,k$. Those are the normal equations of the projection in $L^2$ (with $X=(x_{t-1},\ldots,x_{t-k})'$, $\boldsymbol{\Gamma}=\operatorname{\mathsf{E}}[XX']$ and $\boldsymbol{\gamma}=\operatorname{\mathsf{E}}[Xx_t]$, $\operatorname{\mathsf{E}}[(x_t-a'X)^2]=a'\boldsymbol{\Gamma} a-2a'\boldsymbol{\gamma}+\gamma(0)$, whose gradient vanishes exactly when $\boldsymbol{\Gamma} a=\boldsymbol{\gamma}$), whose solution is unique when the covariance matrix of $(x_{t-1},\ldots,x_{t-k})$ is nonsingular. It is: a singular covariance matrix would make some combination $\sum_{j\leqslant k}b_jx_{t-j}$ with $b\neq\mathbf{0}$ vanish almost surely. Since $\operatorname{cov}(x_{t-j},\varepsilon_{t-l})=\sigma^2\psi_{l-j}$, which is $0$ for $l<j$ and $\sigma^2$ for $l=j$, taking the covariance of that combination with $\varepsilon_{t-1}$ gives $b_1\sigma^2=0$, then with $\varepsilon_{t-2}$ gives $b_2\sigma^2=0$, and so on: the system is triangular and forces every $b_j=0$. ∎

**Remark**. The correlogram and the partial correlogram are complementary identifiers: an $\text{MA}(q)$ has $\rho_k=0$ beyond lag $q$ and a partial correlogram that decays without cutting off, an $\text{AR}(p)$ has $\alpha_{k,k}=0$ beyond lag $p$ and a correlogram that decays without cutting off. Neither cuts off for an $\text{ARMA}$ with both orders positive and $\Phi$, $\Theta$ without a common root. Only the two cut-off statements are proved here (Proposition 10.19 and Theorem 10.23); the statements that the other identifier does not cut off are not.

**Proposition 10.24** (Yule–Walker equations). The coefficients of Definition 10.22 solve the linear system

$$
\rho_l=\sum_{j=1}^{k}\alpha_{j,k}\,\rho_{|l-j|} ,
\qquad l=1,\ldots,k . \tag{19}
$$

**Proof.** The normal equations of the projection in Definition 10.22 (the vanishing of the gradient in the proof of Theorem 10.23) are $\operatorname{\mathsf{E}}[\{x_t-\sum_ja_jx_{t-j}\}\,x_{t-l}]=0$ for $l=1,\ldots,k$, where $x_t=y_t-\mu$; that is $\gamma(l)=\sum_ja_j\gamma(l-j)$, and dividing by $\gamma(0)$, which is positive as a diagonal entry of the nonsingular covariance matrix of Definition 10.22, gives (19). ∎

**Remark**. Two sample versions of $\hat\alpha_{k,k}$ are in use and they are not the same number. Substituting the sample autocorrelations $r_k$ into (19) and solving, which is what `pacf` does, forms every inner product from all $T$ observations; regressing $y_t$ on an intercept and $y_{t-1},\ldots,y_{t-k}$ forms them from the $T-k$ dates at which every regressor exists. The difference is an end effect, of order $T^{-1}$ (not proved here).

**Theorem 10.25** (General $\psi$-weight forecast; FREES §8.5.2). For any process with the representation $y_t=\mu+\sum_{k\geqslant0}\psi_k\varepsilon_{t-k}$ and $\sum_k\psi_k^2<\infty$,

$$
\operatorname{\mathsf{E}}[y_{T+l}\mid\mathcal{L}_T]=\mu+\sum_{k\geqslant l}\psi_k\varepsilon_{T+l-k} ,
\qquad
\operatorname{var}\bigl(y_{T+l}-\operatorname{\mathsf{E}}[y_{T+l}\mid\mathcal{L}_T]\bigr)=\sigma^2\sum_{k=0}^{l-1}\psi_k^2 . \tag{20}
$$

**Proof.** Split the sum at $k=l$. The terms with $k<l$ carry innovations dated after $T$, which are independent of $\mathcal{L}_T$ with mean zero; the terms with $k\geqslant l$ carry innovations dated at or before $T$, which lie in $\mathcal{L}_T$. The forecast error is therefore $\sum_{k<l}\psi_k\varepsilon_{T+l-k}$, a sum of uncorrelated terms. ∎

**Remark** (FREES §8.5.2). FREES writes the forecast-error variance as $\sigma^2\sum_{k=1}^{l-1}\psi_k^2$; the sum starts at $k=0$, as in its own interval and in Theorem 10.25.

**Example 10.4** (Hong Kong exchange rates; FREES §8.6, Tables 8.6–8.8). The daily rates of Unit 9, differenced ($T=501$), with $\text{ARIMA}(2,1,0)$ and $\text{ARIMA}(0,1,2)$ fits:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
hk <- read.csv("data/frees/csv/HKExchange.csv")$EXHKUS; dh <- diff(hk)
round(c(sd_diff = sd(dh), se_rk = 1 / sqrt(length(dh))), 4)
round(acf(dh, lag.max = 10, plot = FALSE)$acf[2:11], 3)
round(pacf(hk, lag.max = 3, plot = FALSE)$acf[, 1, 1], 3)
fa <- arima(dh, order = c(2, 0, 0)); fm <- arima(dh, order = c(0, 0, 2))
tr <- function(f) round(c(coef(f), coef(f) / sqrt(diag(f$var.coef)), s = sqrt(f$sigma2)), 5)
rbind(ar2 = tr(fa), ma2 = tr(fm))
lb <- function(f) sapply(c(2, 4, 6, 8, 10), function(K) Box.test(residuals(f), K, "Ljung-Box")$statistic)
round(rbind(ar2 = lb(fa), ma2 = lb(fm)), 4)

FREES's $\text{AR}(2)$ is reproduced up to rounding: coefficients $0.0899$ and $-0.157$ with $t=2.04$ and $-3.57$ (exact likelihood here), against FREES's $0.0900$ and $-0.158$ with $t=2.03$ and $-3.57$. The $\text{MA}(2)$ gives $t=2.07$ and $-3.47$ ($R$ writes $+\theta$ where FREES writes $-\theta$), against FREES's $2.08$ and $3.66$ in absolute value, so its second $t$ is not reproduced. Both fits give $s=0.00193$, and the Ljung–Box statistics agree with FREES's Table 8.8 to within $0.05$. The partial autocorrelation $0.988$ at lag $1$ of the undifferenced series is FREES's sign of nonstationarity. FREES's Table 8.6 gives $-0.151$ as the lag-$2$ autocorrelation of the differences, which the series reproduces; its text drops the sign. The $-0.158$ is the lag-$2$ partial autocorrelation (FREES's Table 8.7) and FREES's $\text{AR}(2)$ coefficient. FREES's §8.6 quotes the quadratic trend's $R^2=86.2\%$, not the $92.9\%$ of its §7.2 (Unit 9).

**Example 10.5** (An autoregression for NYSE trading volume; ISLR §10.5.2, §10.9.6). ISLR's `NYSE`: daily standardised log trading volume $v_t$, Dow Jones return $r_t$ and log volatility $z_t$, December 1962 to December 1986, trained before 1980 and tested after. Least squares of $v_t$ on an intercept and the five lags of all three series, the lagged design of ISLR (10.21), then with the day of the week, and the “straw man” $\hat v_t=v_{t-1}$; test $R^2=1-\mathrm{MSE}/V_0$ with $V_0$ the test variance:

In [ ]:
xd <- scale(data.matrix(NYSE[, c("DJ_return", "log_volume", "log_volatility")]))
lagm <- function(x, k) rbind(matrix(NA, k, ncol(x)), x[1:(nrow(x) - k), ])
af <- data.frame(log_volume = xd[, "log_volume"], L1 = lagm(xd, 1), L2 = lagm(xd, 2),
                 L3 = lagm(xd, 3), L4 = lagm(xd, 4), L5 = lagm(xd, 5))[-(1:5), ]
itr <- NYSE$train[-(1:5)]; yte <- af[!itr, "log_volume"]; V0 <- var(yte)
r2 <- function(p) 1 - mean((p - yte)^2) / V0
afd <- data.frame(day = NYSE[-(1:5), "day_of_week"], af)
fit5 <- lm(log_volume ~ ., data = af[itr, ]); fitd <- lm(log_volume ~ ., data = afd[itr, ])
round(c(train = sum(itr), test = sum(!itr), ar5 = r2(predict(fit5, af[!itr, ])),
        ar5_day = r2(predict(fitd, afd[!itr, ])), straw_man = r2(af[!itr, "L1.log_volume"])), 3)

ISLR's test $R^2$ of $0.41$ for the autoregression, $0.46$ with the day of the week, and $0.18$ for the straw man are reproduced. The straw man is Unit 9's random-walk forecast without drift; the autoregression is a conditional least squares fit of Definition 10.8 with fifteen lagged regressors, assessed out of sample as in Unit 9.

## 5 Moving Averages and Exponential Smoothing

> **Reading.** FREES §9.1–9.2. SRM learning outcomes 3(b), 3(c).

**Definition 10.26** (Running average; FREES (9.1)). The running average of length $k$ is $\hat s_t=k^{-1}\sum_{j=0}^{k-1}y_{t-j}$, and the forecast it supplies is $\hat y_{T+l}=\hat s_T$ for every $l\geqslant1$.

**Proposition 10.27** (Update and weighted least squares; FREES (9.2)–(9.3)). (i) $\hat s_t=\hat s_{t-1}+(y_t-y_{t-k})/k$.

(ii) $\hat s_T$ is the weighted least squares estimate of $\beta_0$ in $y_t=\beta_0+ \varepsilon_t$ with weights $v_t=\mathbb{1}\{t>T-k\}$.

(iii) Under the model of (ii), $\operatorname{var}(\hat s_t)=\sigma^2/k$.

**Proof.** (i) $k\hat s_t-k\hat s_{t-1}=y_t-y_{t-k}$, the two windows differing in one term at each end. (ii) The weighted criterion is $\sum_{t>T-k}(y_t-b_0)^2$, a quadratic in $b_0$ with derivative $-2\sum_{t>T-k}(y_t-b_0)$, hence minimised at the mean of the last $k$ observations. (iii) The average of $k$ uncorrelated terms. ∎

**Proposition 10.28** (A running average lags a linear trend; FREES §9.1). Let $k\geqslant2$. If $y_t=\beta_0+\beta_1t+\varepsilon_t$ then

$$
\operatorname{\mathsf{E}}[\hat s_t]=\beta_0+\beta_1\Bigl(t-\frac{k-1}{2}\Bigr) ,
\qquad
\operatorname{\mathsf{E}}\bigl[\hat s^{(2)}_t\bigr]=\beta_0+\beta_1\bigl(t-(k-1)\bigr) , \tag{21}
$$

$\hat s^{(2)}$ being the running average of $\hat s^{(1)}=\hat s$. Consequently $2\hat s^{(1)}_T-\hat s^{(2)}_T$ is unbiased for the trend value at $T$ and $b_{1,T}=2(\hat s^{(1)}_T-\hat s^{(2)}_T)/(k-1)$ is unbiased for $\beta_1$.

**Proof.** $\operatorname{\mathsf{E}}[\hat s_t]=k^{-1}\sum_{j<k}\{\beta_0+\beta_1(t-j)\} =\beta_0+\beta_1t-\beta_1k^{-1}\sum_{j<k}j$, and $\sum_{j<k}j=k(k-1)/2$. Applying the same operator again shifts the argument by another $(k-1)/2$. The two combinations follow by solving the pair of linear equations: $2\operatorname{\mathsf{E}}[\hat s^{(1)}]-\operatorname{\mathsf{E}}[\hat s^{(2)}] =\beta_0+\beta_1t$ and $\operatorname{\mathsf{E}}[\hat s^{(1)}]-\operatorname{\mathsf{E}}[\hat s^{(2)}]=\beta_1(k-1)/2$. ∎

**Remark**. Under a linear trend with positive slope, a running average is in expectation too low by $(k-1)/2$ times the slope. Reading the smoothed curve as an estimate of the current level is therefore wrong whenever there is a trend, and double smoothing exists to correct exactly that.

### 5.1 Exponential smoothing

**Definition 10.29** (Exponential smoothing; FREES (9.4)–(9.6)). For $w\in(0,1)$ and a starting value $\hat s_0$,

$$
\hat s_t=(1-w)y_t+w\,\hat s_{t-1} , \tag{22}
$$

and the forecast is $\hat y_{T+l}=\hat s_T$ for every $l\geqslant1$. The smoothing parameter is chosen to minimise the sum of squared one-step errors $\text{SS}(w)=\sum_{t=2}^{T}(y_t-\hat s_{t-1})^2$. From here on $\hat s_t$ (with $\hat s^{(2)}_t$, $b_{0,t}$, $b_{1,t}$ built from it) denotes the exponentially smoothed series, not the running average of Definition 10.26.

**Proposition 10.30** (Unrolled form and discounted least squares; FREES §9.2). (i)

$$
\hat s_T=(1-w)\sum_{j=0}^{T-1}w^{j}y_{T-j}+w^{T}\hat s_0 .
$$

Thus the weights on the observations sum to $1-w^T$ and decay geometrically.

(ii) The weighted least squares estimate of $\beta_0$ in $y_t=\beta_0+\varepsilon_t$ with weights $v_t=w^{T-t}$ is $\sum_{t\leqslant T}w^{T-t}y_t/\sum_{t\leqslant T}w^{T-t}$, which is $\hat s_T$ up to the starting value and the normalisation $1-w^T$.

(iii) Under white noise the estimate in (ii) has variance tending to $\sigma^2(1-w)/(1+w)$ as $T\to\infty$, so smaller $w$ tracks the data and larger $w$ smooths it.

**Proof.** (i) Substitute (22) into itself $T$ times. (ii) Differentiating $\sum_tw^{T-t}(y_t-b_0)^2$ in $b_0$ gives the stated ratio; the denominator is $(1-w^T)/(1-w)$, so the ratio is $(1-w)\sum_jw^jy_{T-j}/(1-w^T)$. (iii) The variance is $\sigma^2\sum_tw^{2(T-t)}/(\sum_tw^{T-t})^2\to\sigma^2(1-w)^2/(1-w^2)$, which is $\sigma^2(1-w)/(1+w)$. ∎

**Theorem 10.31** (The model under which exponential smoothing is optimal (beyond the syllabus)). Let $\{y_t\}$ satisfy $\Delta y_t=\varepsilon_t-\theta\varepsilon_{t-1}$, the $\text{ARIMA}(0,1,1)$ model, with $0<\theta<1$ and a starting value $y_0\in\mathcal{L}_0$ (for instance a constant), so that $y_t\in\mathcal{L}_t$ for $t\geqslant0$. Then the minimum mean squared error one-step forecast satisfies the exponential smoothing recursion with $w=\theta$:

$$
\hat s_T:=\operatorname{\mathsf{E}}[y_{T+1}\mid\mathcal{L}_T]=(1-\theta)y_T+\theta\,\hat s_{T-1} . \tag{23}
$$

**Proof.** Taking conditional expectations in $y_{T+1}=y_T+\varepsilon_{T+1}-\theta\varepsilon_T$ and using $\operatorname{\mathsf{E}}[\varepsilon_{T+1}\mid\mathcal{L}_T]=0$ gives $\hat s_T=y_T-\theta\varepsilon_T$. Applied one step earlier the same identity reads $\hat s_{T-1}=y_{T-1}-\theta \varepsilon_{T-1}$, so the one-step error is

$$
y_T-\hat s_{T-1}=y_T-y_{T-1}+\theta\varepsilon_{T-1}
=\varepsilon_T-\theta\varepsilon_{T-1}+\theta\varepsilon_{T-1}=\varepsilon_T .
$$

Substituting $\varepsilon_T=y_T-\hat s_{T-1}$ into $\hat s_T=y_T-\theta\varepsilon_T$ gives $\hat s_T=y_T-\theta(y_T-\hat s_{T-1})$, which is (23). ∎

**Remark**. Exponential smoothing is therefore not a heuristic: it is the optimal forecast of a nonstationary process whose differences are $\text{MA}(1)$, and the smoothing parameter is that model's moving average coefficient. The same calculation explains the flat forecast function: $\operatorname{\mathsf{E}}[y_{T+l}\mid\mathcal{L}_T]=\hat s_T$ for every $l\geqslant1$, since the extra innovations all have conditional mean zero.

**Proposition 10.32** (Double exponential smoothing; FREES §9.2). Let $\hat s^{(1)}_t=(1-w)y_t+w\hat s^{(1)}_{t-1}$ and $\hat s^{(2)}_t=(1-w)\hat s^{(1)}_t+w\hat s^{(2)}_{t-1}$, started in the infinite past, and let $y_t=\beta_0+\beta_1t+\varepsilon_t$ with $\operatorname{\mathsf{E}}\varepsilon_t=0$. Then

$$
\operatorname{\mathsf{E}}\hat s^{(1)}_t=\beta_0+\beta_1t-\frac{w}{1-w}\beta_1 , \qquad
\operatorname{\mathsf{E}}\hat s^{(2)}_t=\beta_0+\beta_1t-\frac{2w}{1-w}\beta_1 ,
$$

so $b_{0,t}=2\hat s^{(1)}_t-\hat s^{(2)}_t$ and $b_{1,t}=\{(1-w)/w\}(\hat s^{(1)}_t-\hat s^{(2)}_t)$ are unbiased for the level $\beta_0+\beta_1t$ and the slope, and the forecast is $\hat y_{t+l}=b_{0,t}+b_{1,t}\,l$.

**Proof.** With the infinite past, $\hat s^{(1)}_t=(1-w)\sum_{j\geqslant0}w^jy_{t-j}$, so $\operatorname{\mathsf{E}}\hat s^{(1)}_t=(1-w)\sum_jw^j\{\beta_0+\beta_1(t-j)\}=\beta_0+\beta_1t-\beta_1(1-w)\sum_jjw^j$, and $\sum_{j\geqslant0}jw^j=w/(1-w)^2$ (differentiate the geometric series). The same operator applied to the linear function $\operatorname{\mathsf{E}}\hat s^{(1)}_t$ shifts it by another $w\beta_1/(1-w)$. Then $2\operatorname{\mathsf{E}}\hat s^{(1)}_t-\operatorname{\mathsf{E}}\hat s^{(2)}_t=\beta_0+\beta_1t$ and $\operatorname{\mathsf{E}}\hat s^{(1)}_t-\operatorname{\mathsf{E}}\hat s^{(2)}_t=w\beta_1/(1-w)$. ∎

**Proposition 10.33** (Holt's method contains double exponential smoothing; FREES §9.3). Holt's recursions $b_{0,t}=(1-w_1)y_t+w_1(b_{0,t-1}+b_{1,t-1})$ and $b_{1,t}=(1-w_2)(b_{0,t}-b_{0,t-1})+w_2b_{1,t-1}$, started at the double-smoothing values, reproduce the $b_{0,t},b_{1,t}$ of Proposition 10.32 at every $t$ when $w_1=w^2$ and $w_2=2w/(1+w)$.

**Proof.** Write $\alpha=1-w$, $s_t=\hat s^{(1)}_t$, $u_t=\hat s^{(2)}_t$, and use $u_t=\alpha s_t+wu_{t-1}$. Then $b_{0,t}=2s_t-u_t=(1+w)s_t-wu_{t-1}$ and $b_{1,t}=(\alpha/w)(s_t-u_t)=\alpha(s_t-u_{t-1})$. **Level.** From $b_{0,t-1}=2s_{t-1}-u_{t-1}$ and $b_{1,t-1}=(\alpha/w)(s_{t-1}-u_{t-1})$, $b_{0,t-1}+b_{1,t-1}=(2+\alpha/w)s_{t-1}-(1+\alpha/w)u_{t-1}=\{(1+w)s_{t-1}-u_{t-1}\}/w$, since $w+\alpha=1$. So $(1-w^2)y_t+w^2(b_{0,t-1}+b_{1,t-1})=(1+w)\alpha y_t+w(1+w)s_{t-1}-wu_{t-1} =(1+w)s_t-wu_{t-1}=b_{0,t}$. **Slope.** With $1-w_2=\alpha/(1+w)$, $b_{0,t}-b_{0,t-1}=(1+w)s_t-wu_{t-1}-2s_{t-1}+u_{t-1}=(1+w)s_t-2s_{t-1}+\alpha u_{t-1}$, and

$$
\frac{\alpha}{1+w}\{(1+w)s_t-2s_{t-1}+\alpha u_{t-1}\}+\frac{2w}{1+w}\cdot\frac{\alpha}{w}(s_{t-1}-u_{t-1})
=\alpha s_t+\frac{\alpha(\alpha-2)}{1+w}u_{t-1}=\alpha(s_t-u_{t-1}) ,
$$

as $\alpha-2=-(1+w)$; this is $b_{1,t}$. Induction on $t$ from the common start finishes. ∎

**Remark** (FREES §9.3). FREES states the equivalence with $w_1=w_2=2w/(1+w)$; the level parameter is $w^2$, and with FREES's choice the two procedures differ.

**Example 10.6** (Medical component of the CPI; FREES §9.1–9.2, Figures 9.2–9.3). FREES's quarterly index, 1947 to 2007 Q1, as quarterly percentage changes ($T=240$); exponential smoothing started at $\hat s_0=y_1$, and $\text{SS}(w)$ over a grid:

In [ ]:
mc <- read.csv("data/frees/csv/MedCPISmooth.csv"); pm <- mc$PerMEDCPI[-1]
round(c(T = length(pm), first = mc$value[1], last = tail(mc$value, 1),
        quarterly_growth = (tail(mc$value, 1) / mc$value[1])^(1 / 240) - 1), 4)
es <- function(y, w, s0 = y[1]) { s <- numeric(length(y)); prev <- s0
  for (t in seq_along(y)) { s[t] <- (1 - w) * y[t] + w * prev; prev <- s[t] }; s }
SSw <- function(w) { s <- es(pm, w); sum((pm[-1] - head(s, -1))^2) }
c(file_minus_manual = max(abs(es(pm, 0.2) - mc$MCPISMw_2[-1])),
  w_min = optimize(SSw, c(0.01, 0.99))$minimum)
set.seed(3); yh <- cumsum(rnorm(60, 0.5)); w <- 0.7
s1 <- es(yh, w); s2 <- es(s1, w); B0 <- 2 * s1 - s2; B1 <- (1 - w) / w * (s1 - s2)
holt <- function(y, w1, w2) { b0 <- B0[1]; b1 <- B1[1]; out <- matrix(NA, length(y), 2)
  for (t in 2:length(y)) { nb0 <- (1 - w1) * y[t] + w1 * (b0 + b1)
    b1 <- (1 - w2) * (nb0 - b0) + w2 * b1; b0 <- nb0; out[t, ] <- c(b0, b1) }; out }
hc <- holt(yh, w^2, 2 * w / (1 + w)); hf <- holt(yh, 2 * w / (1 + w), 2 * w / (1 + w))
signif(c(holt_w2_minus_double = max(abs(hc[-1, ] - cbind(B0, B1)[-1, ])),
         frees_choice_minus_double = max(abs(hf[-1, ] - cbind(B0, B1)[-1, ]))), 3)

The index rose from $13.3$ to $346.0$, a quarterly growth of $1.37\%$ (FREES $1.36\%$). The file's smoothed series agree with the recursion to its printed digits, and $\text{SS}(w)$ is smallest near $w=0.55$, FREES's “$w\approx0.50$”. Holt's method with $w_1=w^2$ reproduces double smoothing to rounding error; with FREES's $w_1=2w/(1+w)$ it does not. Figure 10.2 shows the smoothed series and $\text{SS}(w)$.

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.6)
op <- par(mfrow = c(1, 2), mar = c(3.4, 3.8, 1.4, 0.6), mgp = c(2.1, 0.6, 0), cex = 0.8)
yr <- mc$YEAR[-1]
plot(yr, pm, type = "l", col = "grey55", xlab = "year", ylab = "percent change",
     main = "w = 0.2 and w = 0.8"); lines(yr, es(pm, 0.2), col = "#1E3A8A", lwd = 1.5)
lines(yr, es(pm, 0.8), col = "#B45309", lwd = 2)
wg <- seq(0.05, 0.95, by = 0.01); plot(wg, sapply(wg, SSw), type = "l", lwd = 2, xlab = "w",
     ylab = "SS(w)", main = "sum of squared one-step errors")
par(op)

*Figure 10.2.* Quarterly percentage changes of the medical CPI with exponential smoothing at $w=0.2$ and $0.8$ (FREES Figure 9.2), and $\text{SS}(w)$ (FREES Figure 9.3).

## 6 Seasonal Time Series Models

> **Reading.** FREES §9.3. SRM learning outcome 3(b).

**Proposition 10.34** (Trigonometric seasonals; FREES (9.7)). $a\sin(ft+b)=c_1\sin(ft)+c_2\cos(ft)$ with $c_1=a\cos b$, $c_2=a\sin b$, so a sum of $m$ harmonics $f_i=2\pi i/s$ (seasonal base $s$) enters a regression as $2m$ explanatory variables. For $s$ even, $m\leqslant s/2$, and at $m=s/2$ the sine column $\sin(\pi t)$ vanishes at integer $t$, so only $s-1$ seasonal columns are available, as many as $s-1$ indicators.

**Proof.** The angle-sum identity $\sin(x+y)=\sin x\cos y+\cos x\sin y$. At $f=2\pi(s/2)/s=\pi$, $\sin(\pi t)=0$ for integer $t$. Harmonics with $i>s/2$ add nothing, because at integer $t$ $\sin\{2\pi(s-i)t/s\}=-\sin(2\pi it/s)$ and $\cos\{2\pi(s-i)t/s\}=\cos(2\pi it/s)$. ∎

**Example 10.7** (Cost of prescription drugs; FREES §9.3 (fit) and Table 9.1 (autocorrelations)). Monthly percentage changes of FREES's cost per claim (Unit 9), $T=67$:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
rx <- read.csv("data/frees/csv/PrescriptionDrug.csv"); yr <- rx$RATEC_C[-1]; tt <- rx$TIME[-1]
m1 <- lm(yr ~ sin(2 * pi * tt / 12) + cos(2 * pi * tt / 12))
m2 <- update(m1, . ~ . + sin(4 * pi * tt / 12) + cos(4 * pi * tt / 12))
round(coef(summary(m1))[, 1:3], 4)
round(c(s_e = summary(m1)$sigma, R2_m1 = summary(m1)$r.squared, R2_m2 = summary(m2)$r.squared,
        partial_F = anova(m1, m2)$F[2],
        F_from_rounded_R2 = ((0.336 - 0.315) / 2) / ((1 - 0.336) / 62)), 4)
round(acf(yr, lag.max = 18, plot = FALSE)$acf[-1], 2)

FREES's fit is reproduced except for the sine coefficient, $-1.6596$, which FREES prints as $-1.6956$; its standard error $0.3269$ and $t=-5.08$ are those of $-1.6596$. The partial $F$ for the second harmonic is $1.014$, $p$ well above $0.05$ (FREES's $0.98$ comes from the rounded $R^2$'s), so $m=1$. The autocorrelations are FREES's Table 9.1 except at lags $7$, $9$ and $15$ ($-0.23$, $-0.08$, $-0.09$, against FREES's $-0.29$, $0.08$, $-0.10$); those at lags $5$–$6$, $10$–$12$ and $17$ exceed $2/\sqrt{67}=0.24$ in absolute value or nearly so.

**Definition 10.35** (Seasonal autoregression; FREES (9.8)). With seasonal base $s$, the $\text{SAR}(P)$ model is

$$
y_t=\beta_0+\beta_1y_{t-s}+\beta_2y_{t-2s}+\cdots+\beta_Py_{t-Ps}+\varepsilon_t , \tag{24}
$$

which omits the intervening lags that an $\text{AR}(Ps)$ would include.

**Proposition 10.36** ($\text{SAR}(1)$ is $s$ interleaved $\text{AR}(1)$ chains; FREES §9.3). Let $\{y_t\}$ be the weakly stationary solution of the $\text{SAR}(1)$ model $y_t=\beta_0+\beta_1y_{t-s}+\varepsilon_t$ with $|\beta_1|<1$. Then each of the $s$ subseries $\{y_{ms+i}\}_{m}$, $i=1,\ldots,s$, is a stationary $\text{AR}(1)$ with coefficient $\beta_1$, the $s$ subseries are independent of one another, and the autocorrelation function of $\{y_t\}$ is

$$
\rho_k=\begin{cases}\beta_1^{k/s} , & s \mid k ,\\[2pt] 0 , & \text{otherwise.}\end{cases} \tag{25}
$$

**Proof.** Reading (24) along a fixed residue class $i$ modulo $s$ gives exactly Definition 10.2 with the innovations $\{\varepsilon_{ms+i}\}_m$, which are independent and identically distributed; different residue classes use disjoint sets of innovations, hence are independent. Theorem 10.3 applied within a class gives $\rho_{ms}=\beta_1^{m}$, and two observations in different classes are independent, so their correlation is $0$. ∎

**Remark**. The correlogram of a $\text{SAR}(1)$ has spikes at $s,2s,\ldots$ and nothing between, which is what distinguishes it from an $\text{AR}(s)$: the latter fills in the intervening lags. A monthly series whose correlogram is large at $12$ and $24$ and negligible elsewhere is asking for the seasonal model, not the twelfth-order one, which would cost eleven extra parameters.

**Example 10.8** (A seasonal autoregression; FREES §9.3). FREES's $\text{SAR}(2)$ with seasonal base $s=6$, by conditional least squares:

In [ ]:
nr <- length(yr); sar <- lm(yr[13:nr] ~ yr[7:(nr - 6)] + yr[1:(nr - 12)])
round(c(coef(summary(sar))[, 1], coef(summary(sar))[, 3], s_e = summary(sar)$sigma), 4)

FREES's $1.2191-0.2867\,y_{t-6}+0.3120\,y_{t-12}$, $t$-statistics $3.00$, $-1.91$, $2.09$ and residual standard deviation $s_e=2.156$ are reproduced; twelve observations are lost to the lags.

**Definition 10.37** (Holt–Winters; FREES §9.3). With smoothing parameters $w_1,w_2,w_3$ and seasonal base $s$, the level, slope and seasonal estimates are updated by

$$
\begin{align}
b_{0,t}&=(1-w_1)(y_t-\hat S_{t-s})+w_1(b_{0,t-1}+b_{1,t-1}) , \tag{26} \\
b_{1,t}&=(1-w_2)(b_{0,t}-b_{0,t-1})+w_2b_{1,t-1} , \tag{27} \\
\hat S_t&=(1-w_3)(y_t-b_{0,t})+w_3\hat S_{t-s} , \tag{28}
\end{align}
$$

and the forecast is $\hat y_{T+l}=b_{0,T}+b_{1,T}l+\hat S_{T+l-s\lceil l/s\rceil}$.

**Proposition 10.38** (Holt–Winters reduces to what it generalises). Dropping (28) and setting $\hat S\equiv0$ leaves Holt's method for a local linear trend; setting in addition $b_{1,t}\equiv0$ and dropping (27) leaves (22) with $w=w_1$.

**Proof.** Immediate substitution. ∎

## 7 Unit Root Tests

> **Reading.** FREES §9.4. SRM learning outcome 3(a).

**Proposition 10.39** (FREES's encompassing model; FREES §9.4). In $y_t=\mu_0+\phi(y_{t-1}-\mu_0)+\mu_1\{\phi+(1-\phi)t\}+\varepsilon_t$: $\phi=1$ gives the random walk $y_t=\mu_1+y_{t-1}+\varepsilon_t$; $\mu_1=0$, $|\phi|<1$ gives the $\text{AR}(1)$ with $\beta_0=\mu_0(1-\phi)$; $\phi=0$ gives the linear trend $\mu_0+\mu_1t+\varepsilon_t$. Subtracting $y_{t-1}$ gives $\Delta y_t=\beta_0+(\phi-1)y_{t-1}+\beta_1t+\varepsilon_t$ with $\beta_0=\mu_0(1-\phi)+\phi\mu_1$ and $\beta_1=\mu_1(1-\phi)$.

**Proof.** Substitute each value of $\phi$ (and $\mu_1$) and collect terms: the right side is $\mu_0(1-\phi)+\phi\mu_1+\phi y_{t-1}+\mu_1(1-\phi)t+\varepsilon_t$. ∎

**Definition 10.40** (Dickey–Fuller regression; FREES (9.9)–(9.10)). Least squares is applied to

$$
\Delta y_t=\beta_0+(\phi-1)\,y_{t-1}+\beta_1t+\varepsilon_t , \tag{29}
$$

and $t_{\text{DF}}$ is the $t$ ratio on $y_{t-1}$. The hypothesis $\phi=1$ is a random walk and $\phi<1$ a stationary alternative, so the test is one-sided to the left. The **augmented** version adds $\sum_{j\leqslant p}c_j\Delta y_{t-j}$ to the right of (29) to absorb serial correlation in the errors.

**Theorem 10.41** (The rate and the shape both change at the boundary). Let $y_t=y_{t-1}+\varepsilon_t$ with $y_0=0$ and $\operatorname{\mathsf{E}}[\varepsilon_t^4]<\infty$, and fit $y_t$ on $y_{t-1}$ without intercept, giving $\hat\phi=\sum_ty_{t-1}y_t/\sum_ty_{t-1}^2$. Then

$$
\frac1T\sum_{t=1}^{T}y_{t-1}\varepsilon_t
=\frac12\Bigl(\frac{y_T^2}{T}-\frac1T\sum_{t=1}^{T}\varepsilon_t^2\Bigr)
\ \xrightarrow{\,d\,}\ \frac{\sigma^2}{2}\,(\chi^2_1-1) , \tag{30}
$$

and

$$
\operatorname{\mathsf{E}}\Bigl[\frac1{T^2}\sum_{t=1}^{T}y_{t-1}^2\Bigr]=\frac{\sigma^2(T-1)}{2T}
\ \longrightarrow\ \frac{\sigma^2}{2} . \tag{31}
$$

So $T(\hat\phi-1)$, not $\sqrt T(\hat\phi-1)$, is the quantity of order one, and its numerator has a limit law that is not normal: it is skewed, with median at $\sigma^2(\chi^2_{1,0.5}-1)/2=-0.273\sigma^2$ and no mass below $-\sigma^2/2$.

**Proof.** From $y_t=y_{t-1}+\varepsilon_t$, squaring gives $y_t^2=y_{t-1}^2+2y_{t-1}\varepsilon_t+\varepsilon_t^2$; summing over $t\leqslant T$ and telescoping gives $y_T^2=2\sum_ty_{t-1}\varepsilon_t+\sum_t\varepsilon_t^2$, which is the first equality in (30). Now $y_T/\sqrt T=T^{-1/2}\sum_t\varepsilon_t\xrightarrow{\,d\,} N(0,\sigma^2)$ by the prerequisites' central limit theorem, so $y_T^2/T\xrightarrow{\,d\,}\sigma^2\chi^2_1$ by the continuous mapping theorem; and $T^{-1}\sum_t\varepsilon_t^2\xrightarrow{\,p\,}\sigma^2$ by the prerequisites' lemma **variance of a stationary average** applied to the iid sequence $\{\varepsilon_t^2\}$, which the fourth moment makes square integrable. Slutsky's theorem combines the two.

For (31), $\operatorname{\mathsf{E}}[y_{t-1}^2]=(t-1)\sigma^2$ and $\sum_{t=1}^{T}(t-1)=T(T-1)/2$.

For the order of $T(\hat\phi-1)=N_T/D_T$, with $N_T=T^{-1}\sum_ty_{t-1}\varepsilon_t$ and $D_T=T^{-2}\sum_ty_{t-1}^2$: by Markov's inequality and (31), $\operatorname{\mathsf{P}}\{D_T>M\}\leqslant\sigma^2/(2M)$. For a lower bound, the Cauchy–Schwarz inequality gives $D_T\geqslant S_T^2$ with $S_T=T^{-3/2}\sum_{t=1}^{T}y_{t-1}=T^{-3/2}\sum_{j=1}^{T-1}(T-j)\varepsilon_j$, whose variance $\sigma^2(T-1)(2T-1)/(6T^2)$ tends to $\sigma^2/3$. Each weight $(T-j)T^{-3/2}$ is at most $T^{-1/2}$ and $\operatorname{var}(S_T)\geqslant\sigma^2/8$ for $T\geqslant2$, so the Lindeberg sum for $S_T/\operatorname{var}(S_T)^{1/2}$ at level $\eta$ is at most $8\sigma^{-2}\operatorname{\mathsf{E}}[\varepsilon_1^2;\,|\varepsilon_1|>c\eta\sqrt T] \leqslant8\sigma^{-2}\operatorname{\mathsf{E}}[\varepsilon_1^4]/(c^2\eta^2T)\to0$, with $c=\sigma/\sqrt8$, and the Lindeberg–Feller theorem of the prerequisites gives $S_T\xrightarrow{\,d\,} N(0,\sigma^2/3)$. Hence $\operatorname{\mathsf{P}}\{D_T<\delta\}\leqslant\operatorname{\mathsf{P}}\{|S_T|<\sqrt\delta\}\to\operatorname{\mathsf{P}}\{|N(0,\sigma^2/3)|<\sqrt\delta\} \leqslant\{6\delta/(\pi\sigma^2)\}^{1/2}$, which is small for small $\delta$. Since $\operatorname{\mathsf{P}}\{|N_T/D_T|>M\}\leqslant\operatorname{\mathsf{P}}\{|N_T|>M\delta\}+\operatorname{\mathsf{P}}\{D_T<\delta\}$ and $\operatorname{\mathsf{P}}\{|N_T/D_T|<\epsilon\}\leqslant\operatorname{\mathsf{P}}\{|N_T|<\epsilon M\}+\operatorname{\mathsf{P}}\{D_T>M\}$, and the limit law in (30) has a continuous distribution function, $T(\hat\phi-1)$ is bounded in probability (for every $\eta>0$ some $M$ gives $\operatorname{\mathsf{P}}\{|T(\hat\phi-1)|>M\}<\eta$ for all large $T$) without tending to $0$ in probability; in particular $\sqrt T(\hat\phi-1)=T^{-1/2}\cdot T(\hat\phi-1)\xrightarrow{\,p\,}0$. ∎

**Remark**. The exact limit law of $t_{\text{DF}}$ is **not** derived here. Obtaining it requires the joint limit of the numerator and the denominator, and the denominator's limit is a functional of a Brownian path, which the prerequisites do not construct. What Theorem 10.41 establishes, in the simplest case (no intercept, no trend), is the two facts that matter for using the test: the estimator converges at rate $T$ rather than $\sqrt T$, and the regression standard error, itself of order $T^{-1}$, adapts to this, so $t_{\text{DF}}$ stays of order one (this last fact is not part of Theorem 10.41 and is not proved here); and the limit of the numerator is not normal, so normal critical values have no justification. With an intercept or a trend, as in Definition 10.40, the analogous facts hold but are not proved here. The critical values are obtained by simulation instead.

**Example 10.9** (Labor force participation rates; FREES §9.4, Table 9.2). The Dickey–Fuller and augmented statistics for FREES's $T=31$ rates, and the $10\%$ points of $t_{\text{DF}}$ under a random walk of the same length, by simulation:

In [ ]:
lf <- read.csv("data/frees/csv/LaborForcePR.csv")$MSC6U; dl <- diff(lf); Tl <- length(lf)
tdf <- function(x, p, trend) { d <- diff(x); n <- length(x); idx <- (p + 2):n
  X <- cbind(ylag = x[idx - 1]); if (trend) X <- cbind(X, t = idx)
  if (p > 0) for (j in 1:p) X <- cbind(X, d[idx - 1 - j])
  coef(summary(lm(d[idx - 1] ~ X)))[2, 3] }
round(rbind(no_trend = sapply(0:2, tdf, x = lf, trend = FALSE),
            trend = sapply(0:2, tdf, x = lf, trend = TRUE)), 3)
set.seed(94)
crit <- t(replicate(20000, { z <- cumsum(rnorm(Tl)); c(tdf(z, 0, FALSE), tdf(z, 0, TRUE)) }))
round(apply(crit, 2, quantile, 0.10), 3)

FREES's six statistics are reproduced; none is below the $10\%$ points, so a random walk is not rejected. The simulated points, $-2.62$ and $-3.21$ at $T=31$, are within $0.03$ of FREES's $-2.624$ and $-3.228$; the normal $10\%$ point $-1.28$ would be badly wrong (the simulation above; Theorem 10.41 shows, for the regression without intercept or trend, why normal theory does not apply).

## 8 ARCH/GARCH Models

> **Reading.** FREES §9.5. SRM learning outcome 3(b).

**Remark** (FREES §9.5). Many financial series show **volatility clustering**: periods of large changes followed by periods of small ones, as in the daily S&P 500 returns of 2000–2006 (FREES Figure 9.10). A constant variance cannot describe this, yet weak stationarity requires one. The models of this section resolve the conflict by letting the variance **conditional on the past**, $\sigma_t^2=\operatorname{var}(\varepsilon_t\mid\varepsilon_{t-1},\varepsilon_{t-2},\ldots)$, change with time while the unconditional variance stays constant. Forecasting the variance, not only the level, matters in finance, for example in option pricing.

**Definition 10.42** (ARCH and GARCH; FREES §9.5). Let $\{z_t\}$ be independent standard normal. The $\text{GARCH}(1,1)$ model is

$$
\varepsilon_t=\sigma_tz_t ,
\qquad
\sigma_t^2=\omega+\alpha\varepsilon_{t-1}^2+\delta\sigma_{t-1}^2 , \tag{32}
$$

with $\omega>0$, $\alpha,\delta\geqslant0$ and $\alpha+\delta<1$, and with $\sigma_t$ a function of $z_{t-1},z_{t-2},\ldots$ (a non-anticipative solution). The case $\delta=0$ is $\text{ARCH}(1)$. FREES writes $w,\gamma_1,\delta_1$ for $\omega,\alpha,\delta$, and $\gamma_1+\delta_1<1$ for $\alpha+\delta<1$.

**Theorem 10.43** (A GARCH series is white noise in the weak sense (uncorrelated) with a non-constant conditional variance; FREES §9.5). Suppose $\{\varepsilon_t\}$ solves (32) and is weakly stationary with $\operatorname{\mathsf{E}}[\varepsilon_t^2]<\infty$. Then

$$
\operatorname{\mathsf{E}}[\varepsilon_t]=0 ,
\qquad
\operatorname{cov}(\varepsilon_t,\varepsilon_{t+k})=0 \ (k\neq0) ,
\qquad
\operatorname{var}(\varepsilon_t)=\frac{\omega}{1-\alpha-\delta} , \tag{33}
$$

while the conditional variance given the past is $\sigma_t^2$, which for $\alpha>0$ is a non-degenerate random variable. Moreover $\{\varepsilon_t^2\}$ satisfies

$$
\varepsilon_t^2=\omega+(\alpha+\delta)\varepsilon_{t-1}^2+\nu_t-\delta\nu_{t-1} ,
\qquad \nu_t=\varepsilon_t^2-\sigma_t^2 , \tag{34}
$$

which has the form of an $\text{ARMA}(1,1)$; if moreover $\operatorname{\mathsf{E}}[\varepsilon_t^4]<\infty$, its innovations $\nu_t$ have mean zero and are uncorrelated; for $\alpha=0$ they are iid, and for $\alpha>0$ they are not independent (not proved here).

**Proof.** $\sigma_t$ is a function of $z_{t-1},z_{t-2},\ldots$ (Definition 10.42), hence is independent of $z_t$, so $\operatorname{\mathsf{E}}[\varepsilon_t]=\operatorname{\mathsf{E}}[\sigma_t]\operatorname{\mathsf{E}}[z_t]=0$ and, for $k>0$, $\operatorname{\mathsf{E}}[\varepsilon_t\varepsilon_{t+k}]=\operatorname{\mathsf{E}}[\varepsilon_t\sigma_{t+k}]\operatorname{\mathsf{E}}[z_{t+k}]=0$, $\varepsilon_t\sigma_{t+k}$ being determined by innovations dated before $t+k$. Taking expectations in (32), $\operatorname{\mathsf{E}}[\varepsilon_t^2]=\operatorname{\mathsf{E}}[\sigma_t^2]$ because $\operatorname{\mathsf{E}}[z_t^2]=1$, so weak stationarity gives $\operatorname{\mathsf{E}}[\varepsilon^2]=\omega+\alpha\operatorname{\mathsf{E}}[\varepsilon^2]+\delta\operatorname{\mathsf{E}}[\varepsilon^2]$ and hence (33). If $\alpha>0$, then $\sigma_t^2=\omega+\sigma_{t-1}^2(\alpha z_{t-1}^2+\delta)$ with $\sigma_{t-1}^2\geqslant\omega>0$ independent of $z_{t-1}$, so $\operatorname{var}(\sigma_t^2\mid\sigma_{t-1}^2)=2\alpha^2\sigma_{t-1}^4>0$ and $\sigma_t^2$ is not degenerate; if $\alpha=0$, then $d_t=\sigma_t^2-\omega/(1-\delta)$ satisfies $d_t=\delta^md_{t-m}$ for every $m\geqslant1$, and $\operatorname{\mathsf{E}}|d_{t-m}|\leqslant\operatorname{\mathsf{E}}[\sigma^2]+\omega/(1-\delta)$ is bounded by weak stationarity, so $\operatorname{\mathsf{E}}|d_t|\leqslant\delta^m\{\operatorname{\mathsf{E}}[\sigma^2]+\omega/(1-\delta)\}\to0$, i.e. $\sigma_t^2=\omega/(1-\delta)$, and $\nu_t=\{\omega/(1-\delta)\}(z_t^2-1)$ is iid.

For (34), substitute $\sigma_t^2=\varepsilon_t^2-\nu_t$ and $\sigma_{t-1}^2=\varepsilon_{t-1}^2-\nu_{t-1}$ into (32):

$$
\varepsilon_t^2-\nu_t=\omega+\alpha\varepsilon_{t-1}^2
 +\delta(\varepsilon_{t-1}^2-\nu_{t-1}) ,
$$

which rearranges to the stated form. That $\nu_t=\sigma_t^2(z_t^2-1)$ has mean zero and is uncorrelated across $t$ follows as above, $\operatorname{\mathsf{E}}[z_t^2-1]=0$ and $\sigma_t^2$ being determined by the past, the fourth moment making each $\nu_t$ square integrable. ∎

**Definition 10.44** ($\text{ARCH}(p)$ and $\text{GARCH}(p,q)$; FREES §9.5). With $\{z_t\}$ independent standard normal and $\varepsilon_t=\sigma_tz_t$, the $\text{GARCH}(p,q)$ model sets

$$
\sigma_t^2=\omega+\sum_{j=1}^{p}\alpha_j\varepsilon_{t-j}^2+\sum_{j=1}^{q}\delta_j\sigma_{t-j}^2 , \tag{35}
$$

with $\omega>0$, $\alpha_j,\delta_j\geqslant0$ and $\sum_j\alpha_j+\sum_j\delta_j<1$, and $\sigma_t$ a function of $z_{t-1},z_{t-2},\ldots$; $q=0$ is $\text{ARCH}(p)$, and $p=q=1$ is Definition 10.42 with $\alpha_1=\alpha$, $\delta_1=\delta$. FREES writes $w$ and $\gamma_j$ for $\omega$ and $\alpha_j$, and $\gamma(1)+\delta(1)<1$ for the condition on the sums. The $\varepsilon_{t-j}^2$ terms let a large shock raise the next variances; the $\sigma_{t-j}^2$ terms make the effect persist, as moving-average terms complement autoregressive ones.

**Proposition 10.45** (The unconditional variance; FREES §9.5). If $\{\varepsilon_t\}$ solves (35) and is weakly stationary with $\operatorname{\mathsf{E}}[\varepsilon_t^2]<\infty$, then $\operatorname{\mathsf{E}}[\varepsilon_t]=0$, the $\varepsilon_t$ are uncorrelated, and

$$
\operatorname{var}(\varepsilon_t)=\frac{\omega}{1-\sum_j\alpha_j-\sum_j\delta_j} .
$$

**Proof.** As in the proof of Theorem 10.43, $\sigma_t$ is independent of $z_t$, which gives the zero mean, the zero correlations and $\operatorname{\mathsf{E}}[\varepsilon_t^2]=\operatorname{\mathsf{E}}[\sigma_t^2]$. Taking expectations in (35) and writing $v$ for the common value of $\operatorname{\mathsf{E}}[\varepsilon_t^2]=\operatorname{\mathsf{E}}[\sigma_t^2]$, $v=\omega+v\sum_j\alpha_j+v\sum_j\delta_j$, so $v=\omega/(1-\sum_j\alpha_j-\sum_j\delta_j)$, positive because the two sums add to less than $1$. ∎

**Theorem 10.46** (Conditional normality produces unconditional fat tails). For a weakly stationary $\text{ARCH}(1)$ with $3\alpha^2<1$ and $\operatorname{\mathsf{E}}[\varepsilon_t^4]$ finite and free of $t$,

$$
\operatorname{\mathsf{E}}[\varepsilon_t^4]=\frac{3\omega^2(1+\alpha)}{(1-\alpha)(1-3\alpha^2)} ,
\qquad
\frac{\operatorname{\mathsf{E}}[\varepsilon_t^4]}{\{\operatorname{\mathsf{E}}[\varepsilon_t^2]\}^2}
=\frac{3(1-\alpha^2)}{1-3\alpha^2}\ >\ 3 \quad\text{for }\alpha>0 . \tag{36}
$$

**Proof.** $\operatorname{\mathsf{E}}[\varepsilon_t^4]=\operatorname{\mathsf{E}}[\sigma_t^4z_t^4]=3\operatorname{\mathsf{E}}[\sigma_t^4]$, the fourth moment of a standard normal being $3$ and $\sigma_t$ independent of $z_t$. With $\delta=0$,

$$
\operatorname{\mathsf{E}}[\sigma_t^4]=\operatorname{\mathsf{E}}[(\omega+\alpha\varepsilon_{t-1}^2)^2]
=\omega^2+2\omega\alpha\operatorname{\mathsf{E}}[\varepsilon^2]+\alpha^2\operatorname{\mathsf{E}}[\varepsilon^4] ,
$$

and $\operatorname{\mathsf{E}}[\varepsilon^2]=\omega/(1-\alpha)$ by (33). Substituting and solving for $\operatorname{\mathsf{E}}[\varepsilon^4]$,

$$
\operatorname{\mathsf{E}}[\varepsilon^4](1-3\alpha^2)=3\omega^2\Bigl(1+\frac{2\alpha}{1-\alpha}\Bigr)
=\frac{3\omega^2(1+\alpha)}{1-\alpha} .
$$

Dividing by $\{\omega/(1-\alpha)\}^2$ gives the kurtosis $3(1+\alpha)(1-\alpha)/(1-3\alpha^2)$, and $1-\alpha^2>1-3\alpha^2>0$ for $\alpha>0$. ∎

**Remark**. The innovations are conditionally normal at every date, and the series is nonetheless leptokurtic: at $\alpha=0.5$ the kurtosis is $9$. Mixing normals of different variances produces heavy tails, as a plot of daily returns shows.

**Definition 10.47** (Conditional likelihood). Given $\varepsilon_1,\ldots,\varepsilon_n$ and a starting value $\sigma_1^2$, the recursion (32) determines $\sigma_2^2,\ldots,\sigma_n^2$, and the parameters are estimated by maximising

$$
\ell(\omega,\alpha,\delta)
=-\frac12\sum_{t=2}^{n}\Bigl\{\ln(2\pi\sigma_t^2)+\frac{\varepsilon_t^2}{\sigma_t^2}\Bigr\} . \tag{37}
$$

**Remark**. The likelihood (37) is not concave in $(\omega,\alpha,\delta)$ in general (not proved here), and its summands are dependent, so the maximum likelihood theorem of the prerequisites does not apply and the standard errors it would supply are not available here. The likelihood is maximised directly and checked against a package; the asymptotic theory of GARCH estimation is not developed here.

**Example 10.10** (S&P 500 daily returns; FREES §9.5, Table 9.3). FREES's model for the $1759$ daily returns of 2000–2006: $y_t=\beta_0+\varepsilon_t-\theta_1 \varepsilon_{t-1}-\theta_2\varepsilon_{t-2}$ with $\text{GARCH}(1,1)$ errors, fitted by maximising the conditional Gaussian likelihood from three starting values:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
spd <- read.csv("data/frees/csv/SP500Daily.csv")
ys <- spd$vwretd[spd$caldt >= 20000101 & spd$caldt <= 20061231]
par(mar = c(4.2, 4, 0.5, 0.5), cex = 0.8)
plot(ys, type = "l", col = "grey30", xlab = "trading day, 2000--2006", ylab = "daily return")
nll <- function(par) { b0 <- par[1]; th1 <- par[2]; th2 <- par[3]; om <- exp(par[4])
  al <- par[5]; de <- par[6]
  if (al < 0 || de < 0 || al + de >= 1) return(1e10)
  e1 <- 0; e2 <- 0; s2 <- var(ys); ll <- 0
  for (t in seq_along(ys)) { e <- ys[t] - b0 + th1 * e1 + th2 * e2
    s2 <- om + al * e1^2 + de * s2; ll <- ll + log(s2) + e^2 / s2; e2 <- e1; e1 <- e }
  0.5 * ll + 0.5 * length(ys) * log(2 * pi) }
starts <- list(c(5e-4, -0.04, -0.06, log(5e-7), 0.07, 0.92),
               c(3e-4, 0, 0, log(1e-6), 0.10, 0.85), c(4e-4, -0.02, -0.05, log(2e-6), 0.05, 0.90))
fits <- lapply(starts, function(st) { o <- optim(st, nll, control = list(maxit = 20000, reltol = 1e-12))
  optim(o$par, nll, method = "BFGS", control = list(maxit = 2000, reltol = 1e-14)) })
best <- fits[[which.min(sapply(fits, `[[`, "value"))]]
print(sapply(fits, function(f) -f$value), digits = 8)
pb <- best$par
signif(c(n = length(ys), beta0 = pb[1], theta1 = pb[2], theta2 = pb[3], omega = exp(pb[4]),
         alpha_on_e2 = pb[5], delta_on_sigma2 = pb[6], logLik = -best$value), 4)

The first and third starts reach the same maximum and the second stops $2.5$ lower, on the boundary $\alpha+\delta=1$ where the penalty halts the search; several starts guard against such stops. The estimates match FREES's Table 9.3 in size: $b_0=0.00046$, $\hat\omega=5.5\times10^{-7}$ (FREES $5.61\times10^{-7}$), the coefficient of $\varepsilon_{t-1}^2$ $0.066$ and that of $\sigma_{t-1}^2$ $0.929$, with log-likelihood $5658.47$ against $5658.85$ (FREES's start of the recursion is not stated). Two labels in FREES are inconsistent with its equation $\sigma_t^2-\delta_1\sigma_{t-1}^2=w+\gamma_1\varepsilon_{t-1}^2$: its table assigns $0.0667$ to $\delta_1$ and $0.9288$ to $\gamma_1$, but the coefficient of $\sigma_{t-1}^2$ is the large one, and its $\theta_1,\theta_2$ carry the opposite sign to its moving average equation. $\hat\alpha+\hat\delta=0.996$ lies near the boundary $\alpha+\delta=1$, where the variance $\omega/(1-\alpha-\delta)$ of Theorem 10.43 grows without bound. The plot, FREES's Figure 9.10, shows the volatility clustering the model is built for: the returns before 2003 swing far more widely than those after, with no trend in their level.

## 9 Regression Trees

> **Reading.** ISLR §8.1.1, to tree pruning (which is Unit 11's), on both reading lists. SRM learning outcomes 4(a), 4(b).

**Definition 10.48** (Partition predictor; ISLR §8.1.1). Let $R_1,\ldots,R_J$ be disjoint sets covering the predictor space. The associated **partition predictor** is

$$
\hat f(\mathbf{x})=\sum_{j=1}^{J}\hat c_j\,\mathbb{1}\{\mathbf{x}\in R_j\} , \tag{38}
$$

constant on each region. A **regression tree** is a partition predictor whose regions are boxes obtained by successive splits on one coordinate at a time.

**Remark** (ISLR §8.1.1). ISLR's vocabulary: the regions are the **terminal nodes** or **leaves** of the tree, the points along the tree where the predictor space is split are its **internal nodes**, and the segments joining the nodes are its **branches**; the tree is drawn upside down, the leaves at the bottom. Not every partition into boxes is a tree: ISLR's Figure 8.3 shows five boxes arranged like a pinwheel, which no sequence of splits of the whole space produces, since the first split of a tree must cut the entire space in two along one coordinate.

**Proposition 10.49** (The region constants are the region means). For fixed regions each containing at least one observation, the values $\hat c_1,\ldots,\hat c_J$ minimising $\sum_{i=1}^{n}\{y_i-\hat f(\mathbf{x}_i)\}^2$ are the region means $\hat c_j=n_j^{-1}\sum_{i:\mathbf{x}_i\in R_j}y_i$, where $n_j=|\{i:\mathbf{x}_i\in R_j\}|$, and the minimised value is $\sum_j\sum_{i\in R_j}(y_i-\hat c_j)^2$.

**Proof.** The criterion separates into $J$ sums, the $j$th involving only $c_j$; each is minimised at the mean of its observations (Unit 2's least squares with an intercept only). ∎

**Proposition 10.50** (A tree is a linear model on indicators). Let $\mathbf{D}$ be the $n\times J$ matrix with $D_{ij}=\mathbb{1}\{\mathbf{x}_i\in R_j\}$. For fixed regions each containing at least one observation ($n_j\geqslant1$ for every $j$), the fitted values of (38) are $\mathbf{D}(\mathbf{D}'\mathbf{D})^{-1}\mathbf{D}'\mathbf{Y}$, the orthogonal projection of $\mathbf{Y}$ on the column space of $\mathbf{D}$, and $\mathbf{D}'\mathbf{D}=\operatorname{diag}(n_1,\ldots,n_J)$.

**Proof.** The regions are disjoint and cover the space, so each row of $\mathbf{D}$ has exactly one $1$ and the columns are orthogonal with squared norms $n_j$. Least squares on $\mathbf{D}$ is Proposition 10.49, and Unit 2 identifies the fitted values with the projection. ∎

**Remark**. The columns of $\mathbf{D}$ are the seasonal indicators of Unit 9 with the calendar replaced by a partition of the predictor space; the identifiability discussion there applies verbatim, so (38) carries no separate intercept. What makes a tree different from an ordinary dummy regression is not the fit given the regions, which is a projection, but that the regions are chosen from the same data.

### 9.1 Recursive binary splitting

**Remark**. Minimising the residual sum of squares over all partitions into $J$ boxes is not a computation that can be carried out. Even the restricted family in which each of $p$ predictors is cut at one of $n-1$ places, applied recursively, has more members than can be enumerated: the first split alone admits $p(n-1)$ choices, and each subsequent split multiplies the count again. Trees are therefore grown greedily, one split at a time.

**Definition 10.51** (Split; ISLR (8.2)–(8.3)). A split of a node $R$ on coordinate $j$ at cutpoint $c$ produces $R_L=R\cap\{\mathbf{x}:x_j<c\}$ and $R_R=R\cap\{\mathbf{x}:x_j\geqslant c\}$. Recursive binary splitting chooses, at each node and among the $(j,c)$ for which $R_L$ and $R_R$ both contain observations, the one minimising

$$
\sum_{i\in R_L}(y_i-\bar y_L)^2+\sum_{i\in R_R}(y_i-\bar y_R)^2 , \tag{39}
$$

and stops when a node has fewer than a preset number of observations.

**Theorem 10.52** (What a split gains). Let a node contain $n$ observations with mean $\bar y$, and let a split send $n_L$ of them to the left with mean $\bar y_L$ and $n_R=n-n_L$ to the right with mean $\bar y_R$, where $n_L,n_R\geqslant1$. Then

$$
\sum_{i}(y_i-\bar y)^2
=\sum_{i\in R_L}(y_i-\bar y_L)^2+\sum_{i\in R_R}(y_i-\bar y_R)^2
 +\frac{n_Ln_R}{n}(\bar y_L-\bar y_R)^2 . \tag{40}
$$

Minimising (39) is therefore the same as maximising $n_Ln_R(\bar y_L-\bar y_R)^2/n$, and the gain is zero exactly when the two region means coincide.

**Proof.** Split the left sum at $\bar y_L$: $\sum_{i\in R_L}(y_i-\bar y)^2 =\sum_{i\in R_L}(y_i-\bar y_L)^2+n_L(\bar y_L-\bar y)^2$, the cross term vanishing because $\sum_{i\in R_L}(y_i-\bar y_L)=0$; similarly on the right. Adding,

$$
\sum_i(y_i-\bar y)^2=\sum_{i\in R_L}(y_i-\bar y_L)^2+\sum_{i\in R_R}(y_i-\bar y_R)^2
 +n_L(\bar y_L-\bar y)^2+n_R(\bar y_R-\bar y)^2 .
$$

Since $n\bar y=n_L\bar y_L+n_R\bar y_R$, one has $\bar y_L-\bar y=(n_R/n) (\bar y_L-\bar y_R)$ and $\bar y_R-\bar y=-(n_L/n)(\bar y_L-\bar y_R)$, so the last two terms are $\{n_Ln_R^2+n_Rn_L^2\}(\bar y_L-\bar y_R)^2/n^2=n_Ln_R(\bar y_L-\bar y_R)^2/n$. ∎

**Corollary 10.53** (Cost of one node). Sorting each predictor once, the best split of a node of size $n$ over $p$ predictors is found in $O(pn)$ further operations.

**Proof.** Sweeping the sorted values of one predictor from left to right, $n_L$ increases by one at a time and $\sum_{i\in R_L}y_i$ is maintained by a single addition, so $\bar y_L$, $\bar y_R$ and the gain of Theorem 10.52 are each computed in a fixed number of operations per candidate cutpoint. There are $n-1$ cutpoints per predictor. ∎

**Remark**. The identity (40) is the one-way analysis of variance decomposition of Unit 3 with two groups. It says a split is judged by the separation of the two group means weighted by $n_Ln_R/n$, so a small group must lie far from the rest to be split off.

**Proposition 10.54** (Monotone invariance). Replacing a predictor $x_j$ by $g(x_j)$ for any strictly increasing $g$ leaves every tree grown by Definition 10.51 unchanged, up to relabelling the cutpoints.

**Proof.** The candidate splits of coordinate $j$ are determined by the ordering of the observed values, which $g$ preserves, and the criterion (39) depends on the split only through which observations fall on each side. ∎

**Remark**. A tree therefore needs no decision about logarithms, standardisation or outlying predictor values, all of which change a linear fit. It pays for that with the two weaknesses of a piecewise-constant predictor: the fitted surface is a step function, so a genuinely linear relation is approximated by a staircase that needs many splits, and the prediction at any $\mathbf{x}$ outside the range of the training data equals the prediction at the nearest point of the box (each coordinate clamped to its observed range, since a cutpoint whose two children are both nonempty satisfies $\min_j<c\leqslant\max_j$, the extremes of the observed $x_j$), so a tree cannot extrapolate at all.

**Example 10.11** (Baseball salaries; ISLR §8.1.1, Figures 8.1–8.2). ISLR's `Hitters` without missing salaries ($n=263$), $\log(\texttt{Salary})$ on `Years` and `Hits`; the tree grown by `tree` and its best three-leaf subtree, which is ISLR's:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
suppressMessages(library(tree)); Hi <- na.omit(Hitters)
tH <- prune.tree(tree(log(Salary) ~ Years + Hits, data = Hi), best = 3); print(tH, digits = 5)
leaf <- ifelse(Hi$Years < 4.5, "R1", ifelse(Hi$Hits < 117.5, "R2", "R3"))
round(rbind(mean_log_salary = tapply(log(Hi$Salary), leaf, mean),
            salary_dollars = 1000 * exp(tapply(log(Hi$Salary), leaf, mean))), 3)
ss <- function(v) sum((v - mean(v))^2); lsal <- log(Hi$Salary); L <- Hi$Years < 4.5
round(c(gain_first_split = ss(lsal) - ss(lsal[L]) - ss(lsal[!L]),
        formula = sum(L) * sum(!L) / length(L) * (mean(lsal[L]) - mean(lsal[!L]))^2), 4)

The splits $\texttt{Years}<4.5$ and $\texttt{Hits}<117.5$ give leaf means $5.107$, $5.998$ and $6.740$, and ISLR's prediction $e^{5.107}$ thousand dollars, \$165,174, for the first leaf (\$165,139 from the unrounded mean). The gain of the first split equals $n_Ln_R(\bar y_L-\bar y_R)^2/n$, Theorem 10.52.

## 10 Computation in R

> **Reading.** FREES chapters 8–9 scripts; ISLR §10.9.6 (beyond the syllabus).

**Example 10.12** (Conditional least squares four ways; Definition 10.8). On the inflation bond index: `lm`, the closed form of Proposition 10.9, `arima` with `method = "CSS"`, and the approximation $r_1$:

In [ ]:
ym <- yb[-Tb]; yp <- yb[-1]
b1_closed <- sum((ym - mean(ym)) * (yp - mean(yp))) / sum((ym - mean(ym))^2)
acss <- arima(yb, order = c(1, 0, 0), method = "CSS")
print(c(lm = coef(mb)[[2]], closed_form = b1_closed, arima_css = coef(acss)[["ar1"]],
        r1 = rb[1], arima_default = coef(arima(yb, order = c(1, 0, 0)))[["ar1"]]), digits = 8)

All four agree to within $10^{-5}$ here; the end effect of Proposition 10.9 shows in $b_0$ (Example 10.2: least squares $0.4085$ against $\bar y(1-r_1)=0.4179$), and the default `arima`, which maximises the exact likelihood, differs again.

**Example 10.13** (Partial autocorrelations from the Yule–Walker system; Proposition 10.24).

In [ ]:
ra <- acf(yb, lag.max = 3, plot = FALSE)$acf[2:4]
yw <- sapply(1:3, function(k) { R <- toeplitz(c(1, ra)[1:k]); solve(R, ra[1:k])[k] })
reg <- sapply(1:3, function(k) { X <- sapply(1:k, function(j) yb[(k + 1 - j):(Tb - j)])
  tail(coef(lm(yb[(k + 1):Tb] ~ X)), 1) })
round(rbind(yule_walker = yw, pacf = pacf(yb, lag.max = 3, plot = FALSE)$acf[, 1, 1],
            lagged_regression = unname(reg)), 4)

`pacf` solves the Yule–Walker system in the sample autocorrelations; the lagged regression uses only the dates at which every lag exists, and differs by an end effect.

**Example 10.14** (Portmanteau statistics; Theorem 10.13). Ljung–Box by hand against `Box.test`, and the null distribution for white noise and for the residuals of a fitted $\text{AR}(1)$, $T=100$, $K=10$, $5000$ replications:

In [ ]:
eh <- residuals(fa)[-(1:2)]; Kl <- 10; nT <- length(eh)
rk <- acf(eh, lag.max = Kl, plot = FALSE)$acf[-1]
c(by_hand = nT * (nT + 2) * sum(rk^2 / (nT - 1:Kl)),
  Box.test = unname(Box.test(eh, Kl, "Ljung-Box")$statistic))
set.seed(105)
q <- replicate(5000, { z <- arima.sim(list(ar = 0.6), 100)
  c(white = Box.test(rnorm(100), 10, "Ljung-Box")$statistic,
    ar1_resid = Box.test(residuals(arima(z, order = c(1, 0, 0))), 10, "Ljung-Box")$statistic) })
round(rbind(mean = rowMeans(q), size_chisq10 = rowMeans(q > qchisq(0.95, 10)),
            size_chisq9 = rowMeans(q > qchisq(0.95, 9))), 3)

For white noise the mean is near $10$ and $\chi^2_{10}$ rejects $6.3\%$ of the time at $T=100$; for fitted residuals the mean is near $9$ and only $\chi^2_{K-p}=\chi^2_9$ keeps the size near $5\%$.

## 11 Exercises

**Exercise 10.1** (FREES exercise 8.3). Using Unit 2's simple regression with $y_{t-1}$ in place of $x_t$, $t=2,\ldots,T$: (a) give $b_1$ exactly; (b) give $b_0$ exactly; (c) show $b_0\approx\bar y(1-r_1)$.

**Solution.** (a), (b) Proposition 10.9. (c) The same proposition: $b_1\approx r_1$ and $\bar y_\pm\approx\bar y$ up to end effects, so $b_0=\bar y_+-b_1\bar y_-\approx\bar y(1-r_1)$. On the inflation bond index, $b_0=0.4085$ against $\bar y(1-r_1)=0.4179$ (Example 10.2). ■

**Exercise 10.2** (FREES exercise 8.4). For the $\text{AR}(1)$ model show (a) $\sigma_y^2(1-\beta_1^2)=\sigma^2$; (b) $\operatorname{cov}(y_t,y_{t-1})=\beta_1\sigma_y^2$; (c) $\operatorname{cov}(y_t,y_{t-k})=\beta_1^k\sigma_y^2$; (d) $\rho_k=\beta_1^k$.

**Solution.** Theorem 10.3. Directly: (a) $\varepsilon_t$ is uncorrelated with $y_{t-1}$, so taking variances in (1), $\sigma_y^2=\beta_1^2\sigma_y^2+\sigma^2$. (b), (c) Multiply $y_t-\mu=\beta_1(y_{t-1}-\mu)+\varepsilon_t$ by $y_{t-k}-\mu$ and take expectations: $\gamma(k)=\beta_1\gamma(k-1)$, since $\varepsilon_t$ is uncorrelated with $y_{t-k}$ for $k\geqslant1$; iterate from $\gamma(0)=\sigma_y^2$. (d) Divide by $\sigma_y^2$. ■

**Exercise 10.3** (FREES exercise 8.5). For the $\text{AR}(1)$ model, (a) use the chain rule to show $y_{T+k}-\hat y_{T+k}\approx\varepsilon_{T+k}+\beta_1\varepsilon_{T+k-1}+\cdots+\beta_1^{k-1}\varepsilon_{T+1}$; (b) show that its variance is about $\sigma^2\sum_{l=0}^{k-1}\beta_1^{2l}$.

**Solution.** Theorem 10.15: with the parameters known the chain rule gives $\hat y_{T+k}=\mu+\beta_1^k(y_T-\mu)$ and the error is exactly $\sum_{l<k}\beta_1^l\varepsilon_{T+k-l}$, of variance $\sigma^2\sum_{l<k}\beta_1^{2l}$; with estimated parameters the error differs by a term of order $T^{-1/2}$ (Theorem 10.10 and its proof), hence “$\approx$”. ■

**Exercise 10.4** (FREES exercise 8.6, part (c)). For the $503$ daily returns of the S&P value-weighted index in 2005–2006 (Unit 9), compute the autocorrelations at lags $1$ to $10$ and say which differ significantly from zero.

**Solution.**

In [ ]:
r56 <- spd$vwretd[spd$caldt >= 20050101 & spd$caldt <= 20061231]
rr <- acf(r56, lag.max = 10, plot = FALSE)$acf[-1]
round(rbind(r_k = rr, z = rr * sqrt(length(r56))), 3)
round(c(band = 2 / sqrt(length(r56)), Q_LB = unname(Box.test(r56, 10, "Ljung-Box")$statistic),
        p = Box.test(r56, 10, "Ljung-Box")$p.value), 4)

By Corollary 10.7, under white noise each $|r_k|$ exceeds $2/\sqrt{503}=0.089$ with probability about $0.045$. Only $r_2=-0.096$ ($z=-2.15$) crosses the band, about what ten looks at white noise would produce, and the Ljung–Box statistic for the ten lags jointly, $10.38$, gives $p=0.41$ against $\chi^2_{10}$ (Theorem 10.13): no autocorrelation is detected, as the market efficiency argument of FREES's exercise expects. ■

**Exercise 10.5** (ISLR §8.4, exercise 1). Draw a partition of a two-dimensional predictor space that recursive binary splitting could produce, with at least six regions, and the corresponding tree, labelling regions and cutpoints.

**Solution.** Split $X_1<t_1$; on $\{X_1<t_1\}$ split $X_2<t_2$, giving $R_1$ below and, above, split $X_1<t_3$ ($t_3<t_1$) into $R_2$ and $R_3$; on $\{X_1\geqslant t_1\}$ split $X_2<t_4$, giving $R_4$ above and, below, split $X_1<t_5$ ($t_5>t_1$) into $R_5$ and then $X_2<t_6$ on the rest into $R_6$, $R_7$. Every region is a box obtained by successive one-coordinate splits (Definition 10.51); Figure 10.3 draws both:

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 2.4)
op <- par(mfrow = c(1, 2), mar = c(2.4, 2.4, 0.6, 0.6), mgp = c(1.3, 0.4, 0), cex = 0.75)
plot(NA, xlim = c(0, 10), ylim = c(0, 10), xlab = "X1", ylab = "X2", xaxt = "n", yaxt = "n")
segments(5, 0, 5, 10); segments(0, 4, 5, 4); segments(2, 4, 2, 10); segments(5, 6, 10, 6)
segments(7, 0, 7, 6); segments(7, 3, 10, 3)
text(c(2.5, 1, 3.5, 7.5, 6, 8.5, 8.5), c(2, 7, 7, 8, 3, 1.5, 4.5),
     c("R1", "R2", "R3", "R4", "R5", "R6", "R7"))
axis(1, at = c(2, 5, 7), labels = c("t3", "t1", "t5")); axis(2, at = c(3, 4, 6), labels = c("t6", "t2", "t4"), las = 1, cex.axis = 0.7)
plot(NA, xlim = c(0, 10), ylim = c(0, 4.2), axes = FALSE, xlab = "", ylab = "")
segments(c(2.5, 2.5, 7.5, 1.25, 1.25, 3.75, 6.25, 6.25, 8.75, 5.5, 5.5, 7, 6.25, 7.75),
         c(4, 4, 4, 3, 3, 3, 3, 3, 3, 2, 2, 2, 1, 1),
         c(2.5, 7.5, 7.5, 1.25, 3.75, 3.75, 6.25, 8.75, 8.75, 5.5, 7, 7, 6.25, 7.75),
         c(3, 4, 3, 2, 3, 2, 2, 3, 2, 1, 2, 1, 0.3, 0.3))
segments(c(3, 6.25), c(2, 1), c(4.5, 7.75), c(2, 1))
text(c(5, 2.5, 7.5, 3.75, 6.25, 7), c(4.15, 3.15, 3.15, 2.15, 2.15, 1.15),
     c("X1 < t1", "X2 < t2", "X2 < t4", "X1 < t3", "X1 < t5", "X2 < t6"), cex = 0.8)
text(c(1.25, 3, 4.5, 8.75, 5.5, 6.25, 7.75), c(1.8, 1.8, 1.8, 1.8, 0.8, 0.1, 0.1),
     c("R1", "R2", "R3", "R4", "R5", "R6", "R7"), cex = 0.8)
par(op)

*Figure 10.3.* A seven-region partition from recursive binary splitting and its tree. ■

**Exercise 10.6** (ISLR §8.4, exercise 4). (a) Sketch the tree for the partition of ISLR Figure 8.14 (left): regions with means $15$ on $\{X_1<1,X_2\geqslant1\}$, $3$ on $\{X_1<0,X_2<1\}$, $0$ on $\{0\leqslant X_1<1,0\leqslant X_2<1\}$, $10$ on $\{0\leqslant X_1<1,X_2<0\}$ and $5$ on $\{X_1\geqslant1\}$. (b) Draw the partition for the tree of Figure 8.14 (right): $X_2<1$, then $X_1<1$ with leaves $-1.80$ and $0.63$; otherwise $X_2<2$, then $X_1<0$ with leaves $-1.06$ and $0.21$, and leaf $2.49$ for $X_2\geqslant2$.

**Solution.** (a) Root $X_1<1$: right leaf $5$. Left: $X_2<1$; its right child is the leaf $15$; its left child splits $X_1<0$ into the leaf $3$ and a node $X_2<0$ with leaves $10$ ($X_2<0$) and $0$. (b) The line $X_2=1$ divides the plane; below it, $X_1=1$ separates $-1.80$ (left) from $0.63$; above it, $X_2=2$ separates the strip $1\leqslant X_2<2$, cut by $X_1=0$ into $-1.06$ and $0.21$, from the half-plane $X_2\geqslant2$ with mean $2.49$. Each leaf mean is the region mean (Proposition 10.49). ■

**Exercise 10.7** (ISLR §10.10, exercise 9; beyond the syllabus). Fit the lag-5 autoregression to `NYSE`, and refit it with a twelve-level month factor. Does the month improve the test $R^2$?

**Solution.**

In [ ]:
options(repr.plot.width = 6.4, repr.plot.height = 3)
mon <- factor(substr(as.character(NYSE$date), 6, 7))[-(1:5)]
afm <- data.frame(month = mon, af); fitm <- lm(log_volume ~ ., data = afm[itr, ])
round(c(ar5 = r2(predict(fit5, af[!itr, ])), ar5_month = r2(predict(fitm, afm[!itr, ]))), 4)

The month factor changes the test $R^2$ in the third decimal only; the day of the week (Example 10.5) matters, the month does not. ■

**Exercise 10.8** (ISLR §10.10, exercise 10; beyond the syllabus). Fit the linear autoregression by “flattening” the $5\times3$ array of lagged values (ISLR §10.9.6 builds it as the input of a recurrent network), and compare its test $R^2$ with the `lm` fit.

**Solution.** The flattened array of the five lagged 3-vectors is the same set of fifteen regressors in another order, so least squares gives the same fit (Unit 2: fitted values depend on the column space only):

In [ ]:
xr <- array(data.matrix(af[, -1]), c(nrow(af), 3, 5))[, , 5:1]
flat <- matrix(xr, nrow(af)); fitf <- lm(af$log_volume[itr] ~ flat[itr, ])
pf <- cbind(1, flat[!itr, ]) %*% coef(fitf)
round(c(flattened = r2(pf), lm = r2(predict(fit5, af[!itr, ]))), 6)

The two are identical. `lm` on a data frame keeps names and handles factors such as the day of the week; the flattened array is the input format ISLR's recurrent network needs. ■

**Exercise 10.9** (SOA sample question 22). For the stationary $\text{AR}(1)$ $y_t=\beta_0+\beta_1y_{t-1}+\varepsilon_t$, which statement is NOT true? (A) $\beta_0$ must not equal $1$. (B) $|\beta_1|<1$. (C) If $\beta_1=0$ the model is white noise. (D) If $\beta_1=1$ it is a random walk. (E) Only $y_{t-1}$ is used as a predictor for $y_t$.

**Solution.** Theorem 10.3 needs $|\beta_1|<1$ (B) and puts no restriction on $\beta_0$, so (A) is false. (C), (D) are Proposition 10.4, and (E) is the definition. The answer is (A). ■

**Exercise 10.10** (SOA sample question 58). For $y_1,\ldots,y_6=31,35,37,41,45,51$ from an $\text{AR}(1)$, the approximation to conditional least squares is used. Calculate $s^2$: (A) 13; (B) 21; (C) 22; (D) 26; (E) 35.

**Solution.** The approximation of Proposition 10.9 is $b_1=r_1$, $b_0=\bar y(1-r_1)$:

In [ ]:
y6 <- c(31, 35, 37, 41, 45, 51); a6 <- y6 - mean(y6)
r1s <- sum(a6[-1] * a6[-6]) / sum(a6^2); b0s <- mean(y6) * (1 - r1s)
e6 <- y6[-1] - (b0s + r1s * y6[-6])
round(c(r1 = r1s, b0 = b0s, mean_e = mean(e6), s2 = sum((e6 - mean(e6))^2) / (6 - 3),
        s2_least_squares = summary(lm(y6[-1] ~ y6[-6]))$sigma^2), 4)

$r_1=117/262=0.4466$, $b_0=22.137$, the residuals average $2.78$ (not $0$, as the fit is not least squares) and (6) gives $s^2=21.97$: (C). Conditional least squares itself gives a much smaller $s^2$. ■

**Exercise 10.11** (SOA sample question 64). For a stationary $\text{AR}(1)$ $y_t=\beta_0+\beta_1y_{t-1}+\varepsilon_t$, which is always true? (A) $\beta_0\neq0$; (B) $\beta_0=1$; (C) $\beta_1=0$; (D) $\beta_1=1$; (E) $|\beta_1|<1$.

**Solution.** Theorem 10.3 and Proposition 10.4: stationarity requires $|\beta_1|<1$, and $\beta_0$ is any constant: (E). ■

**Exercise 10.12** (SOA sample question 46). A series observed at $t=0,\ldots,100$ ends with $y_{97},\ldots,y_{100}=96.9,98.1,99.0,100.2$, with $\hat s^{(1)}_{97},\hat s^{(1)}_{98},\hat s^{(1)}_{99}=93.1,94.1,95.1$ and $\hat s^{(2)}_{97},\hat s^{(2)}_{98}=88.9,89.9$ at $w=0.8$. With $F$ the exponential smoothing and $G$ the double exponential smoothing forecast of $y_{102}$ (forecasts to one decimal, the trend to three), calculate $|F-G|$: (A) 0.0; (B) 2.1; (C) 4.2; (D) 6.3; (E) 8.4.

**Solution.** By Definition 10.29, $\hat s^{(1)}_{100}=0.2(100.2)+0.8(95.1)=96.1$, the forecast at every lead, so $F=96.1$. Then $\hat s^{(2)}_{99}=0.2(95.1)+0.8(89.9)=90.9$ and $\hat s^{(2)}_{100}=0.2(96.1)+0.8(90.9)=91.9$. By Proposition 10.32, $b_{1,100}=(0.2/0.8)(96.1-91.9)=1.05$ and $b_{0,100}=2(96.1)-91.9=100.3$, so $G=100.3+2(1.05)=102.4$ and $|F-G|=6.3$: (D). ■